# Fusion des fichiers excel de 2021 à 2025 UNIQUEMENT les feuilles 1

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
PIPELINE SOLDE - PARTIE 1 : FUSION FEUILLES 1
Version finale avec détection d'en-tête flexible
"""

import boto3
import io
import re
import unicodedata
from io import BytesIO
from collections import defaultdict
import pandas as pd
import numpy as np

# =============================================================================
# NORMALIZER
# =============================================================================
class Normalizer:
    """Fonctions de normalisation de texte."""

    @staticmethod
    def normalize_label(s: str) -> str:
        """Normalisation complète pour le matching."""
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return ""
        s = str(s).strip()
        
        # Retirer préfixes
        s = re.sub(r'\b[A-Z][A-Z_0-9]*\d{6}\.', '', s)
        s = re.sub(r'\|\|[A-Z][A-Z_0-9]*\d{6}\.', '||', s)
        
        # Normalisation standard
        s = (s.replace("-", " ")
               .replace("'", "'")
               .replace("'", " ")
               .replace(".", " "))
        s = s.lower()
        s = unicodedata.normalize("NFKD", s)
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        s = re.sub(r"\s+", " ", s).strip()
        return s

    @staticmethod
    def normalize_column_name(col: str) -> str:
        """Normalise les noms de colonnes Excel."""
        col = str(col)

        if '||' in col and '.' in col:
            parties = col.split('||')
            parties_sans_prefixe = [p.split('.')[-1] for p in parties]
            col = '||'.join(parties_sans_prefixe)
        else:
            col = col.split('.')[-1] if '.' in col else col

        placeholder = "###CODEORGANISME###"
        col = col.replace("CODE_ORGANISME", placeholder).replace("CODE ORGANISME", placeholder)
        col = col.replace('_', ' ')
        col = re.sub(r'\s+', ' ', col)
        col = re.sub(r'\s*/\s*', '/', col)
        col = re.sub(r'\s*\|\|\s*', '||', col)
        col = col.replace(placeholder, "CODE_ORGANISME").strip()

        if col == "IMPOT":
            col = "IMPOT "
        if col == "RETENUE PENSION":
            col = "RETENUE  PENSION"

        return col

    @staticmethod
    def normalize_matricule(s: pd.Series) -> pd.Series:
        """Normalise les matricules."""
        return (s.astype("string")
                 .str.strip()
                 .str.replace("\u00A0", "", regex=False)
                 .str.upper())


# =============================================================================
# CONNEXION S3
# =============================================================================
print("="*80)
print("🚀 PIPELINE SOLDE - PARTIE 1 : FUSION FEUILLES 1")
print("="*80 + "\n")

print("⚙️  Connexion à MinIO/S3...")
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"
prefixes = [
    "Solde/DONNEES 2021/",
    "Solde/DONNEES 2022/",
    "Solde/DONNEES 2023/",
    "Solde/DONNEES 2024/",
    "Solde/DONNEES 2025/",
]

print("✅ Connexion établie\n")

# =============================================================================
# LISTING FICHIERS
# =============================================================================
print("📋 Recensement des fichiers Excel...")

fichiers_par_mois = defaultdict(list)

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            if key.split('/')[-1].startswith('~$'):
                continue
            match = re.search(r"(\d{6})\.xlsx$", key)
            if match:
                mois_annee = match.group(1)
                fichiers_par_mois[mois_annee].append(key)

print(f"✅ {len(fichiers_par_mois)} périodes trouvées\n")

# Contrôle complétude annuelle
print("📊 Vérification complétude annuelle:")
fichiers_par_annee = defaultdict(set)
for mois_annee in fichiers_par_mois.keys():
    if len(mois_annee) == 6:
        mois = mois_annee[:2]
        annee = mois_annee[2:]
        fichiers_par_annee[annee].add(mois)

for annee, mois_set in sorted(fichiers_par_annee.items()):
    mois_complets = set(f"{i:02d}" for i in range(1, 13))
    mois_manquants = sorted(mois_complets - mois_set)
    if mois_manquants:
        print(f"⚠️  Année {annee}: {len(mois_set)}/12 fichiers (manquants: {', '.join(mois_manquants)})")
    else:
        print(f"✅ Année {annee}: 12/12 fichiers")

print()

# =============================================================================
# COLONNES ATTENDUES (AVEC COLONNES CRITIQUES)
# =============================================================================

# ✅ COLONNES CRITIQUES (doivent être présentes)
colonnes_critiques = {
    "MATRICULE||CODE_ORGANISME",
    "MATRICULE",
    "NOM",
    "MONTANT BRUT",
}

# Colonnes importantes mais non-bloquantes
colonnes_importantes = {
    "DATE NAISSANCE", "SEXE", "SITUATION MATRIMONIALE", "NOMBRE ENFANT",
    "LIEU AFFECTATION", "SERVICE", "ORGANISME", "PRISE DE SERVICE",
    "SITUATION", "DATE DEBUT SITUATION", "EMPLOI", "FONCTION", "GRADE",
    "CLASSE/ECHELON", "MONTANT NET", "RETENUE  PENSION", "IMPOT ",
    "CHARGE PATRONALE", "MOIS/ANNEE", "GRADE 2", "AGE RETRAITE", "DATE RETRAITE"
}

colonnes_critiques_norm = {Normalizer.normalize_label(c) for c in colonnes_critiques}
colonnes_importantes_norm = {Normalizer.normalize_label(c) for c in colonnes_importantes}

print("🔍 Stratégie de détection:")
print(f"   • Colonnes CRITIQUES (obligatoires): {len(colonnes_critiques)}")
print(f"   • Colonnes IMPORTANTES (optionnelles): {len(colonnes_importantes)}")
print(f"   • Seuil minimum: 100% critiques + 50% importantes\n")

# =============================================================================
# CHARGEMENT + FUSION AVEC SEUIL
# =============================================================================

print("🔄 Chargement et fusion des fichiers...\n")

resultat_final = []
stats = {
    'succes': 0,
    'echec_header': 0,
    'echec_colonnes': 0,
    'echec_autre': 0
}

for mois, fichiers in sorted(fichiers_par_mois.items()):
    for fichier in fichiers:
        filename = fichier.split('/')[-1]
        
        try:
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj["Body"].read()

            df_raw = pd.read_excel(BytesIO(data_bytes), sheet_name=0, header=None)
            df_raw.dropna(how="all", inplace=True)

            if df_raw.empty:
                print(f"⚠️  {filename}: fichier vide")
                stats['echec_autre'] += 1
                continue

            # ✅ DÉTECTION AVEC SEUIL
            header_row_index = None
            best_score = 0
            
            for i, row in df_raw.iterrows():
                if i > 50:  # Limiter le scan
                    break
                    
                colonnes_ligne = row.astype(str).tolist()
                colonnes_normalisees = {Normalizer.normalize_label(col) for col in colonnes_ligne}
                
                # Calculer scores
                critiques_presentes = colonnes_critiques_norm & colonnes_normalisees
                importantes_presentes = colonnes_importantes_norm & colonnes_normalisees
                
                score_critiques = len(critiques_presentes) / len(colonnes_critiques_norm)
                score_importantes = len(importantes_presentes) / len(colonnes_importantes_norm)
                
                # Score global (critiques pèsent plus lourd)
                score_total = (score_critiques * 0.7) + (score_importantes * 0.3)
                
                # Si toutes les critiques présentes + au moins 50% des importantes
                if score_critiques == 1.0 and score_importantes >= 0.5:
                    if score_total > best_score:
                        best_score = score_total
                        header_row_index = i

            if header_row_index is None:
                print(f"❌ {filename}: en-tête non détectée (meilleur score: {best_score:.2f})")
                stats['echec_header'] += 1
                continue

            # Relire avec le bon en-tête
            df = pd.read_excel(
                BytesIO(data_bytes),
                sheet_name=0,
                skiprows=header_row_index
            )

            df.dropna(how="all", inplace=True)

            if df.empty:
                print(f"⚠️  {filename}: vide après nettoyage")
                stats['echec_autre'] += 1
                continue

            # Normaliser noms de colonnes
            df.columns = [Normalizer.normalize_column_name(c) for c in df.columns]

            # ✅ VÉRIFIER SEULEMENT LES COLONNES CRITIQUES
            colonnes_presentes = set(df.columns)
            critiques_manquantes = colonnes_critiques - colonnes_presentes
            
            if critiques_manquantes:
                print(f"❌ {filename}: colonnes CRITIQUES manquantes: {critiques_manquantes}")
                stats['echec_colonnes'] += 1
                continue
            
            # Vérifier colonnes importantes manquantes (warning seulement)
            importantes_manquantes = colonnes_importantes - colonnes_presentes
            if importantes_manquantes:
                print(f"⚠️  {filename}: {len(importantes_manquantes)} colonnes optionnelles manquantes")

            # Ajouter métadonnées
            df["PERIODE"] = mois
            df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y") + pd.offsets.MonthEnd(0)
            df["SOURCE_FICHIER"] = filename

            resultat_final.append(df)
            stats['succes'] += 1
            
            print(f"✅ {filename}: {len(df):,} lignes")

        except Exception as e:
            print(f"❌ {filename}: erreur - {e}")
            stats['echec_autre'] += 1
            continue

# =============================================================================
# STATISTIQUES ET EXPORT
# =============================================================================

print("\n" + "="*80)
print("📊 STATISTIQUES")
print("="*80)
print(f"   • Fichiers traités avec succès  : {stats['succes']}")
print(f"   • Échec détection en-tête       : {stats['echec_header']}")
print(f"   • Échec colonnes manquantes     : {stats['echec_colonnes']}")
print(f"   • Autres erreurs                : {stats['echec_autre']}")
print(f"   • TOTAL                         : {sum(stats.values())}")

if resultat_final:
    print("\n🔗 Concaténation finale...")
    panel_solde_df = pd.concat(resultat_final, ignore_index=True)
    panel_solde_df.dropna(how="all", inplace=True)

    print(f"✅ Panel créé: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")

    print("\n📅 Conversion des dates...")
    colonnes_dates = ["DATE NAISSANCE", "PRISE DE SERVICE", "DATE DEBUT SITUATION", 
                     "DATE RETRAITE", "DATE_COLLECTE"]
    
    for col in colonnes_dates:
        if col in panel_solde_df.columns:
            panel_solde_df[col] = pd.to_datetime(panel_solde_df[col], errors="coerce")

    print("\n🔢 Conversion des valeurs numériques...")
    colonnes_numeriques = ["NOMBRE ENFANT", "MONTANT BRUT", "MONTANT NET",
                          "RETENUE  PENSION", "IMPOT ", "CHARGE PATRONALE",
                          "MOIS/ANNEE", "AGE RETRAITE"]
    
    for col in colonnes_numeriques:
        if col in panel_solde_df.columns:
            panel_solde_df[col] = pd.to_numeric(panel_solde_df[col], errors="coerce")

    # ✅ CORRECTION: FORCER LES COLONNES OBJECT EN STRING
    print("\n🧹 Nettoyage des types pour export Parquet...")
    
    # Identifier toutes les colonnes object
    object_cols = panel_solde_df.select_dtypes(include=['object']).columns
    
    for col in object_cols:
        try:
            # Convertir en string, puis remplacer les valeurs vides par NA
            panel_solde_df[col] = panel_solde_df[col].astype(str)
            panel_solde_df[col] = panel_solde_df[col].replace(['nan', 'None', '<NA>', ''], pd.NA)
            panel_solde_df[col] = panel_solde_df[col].astype("string")
            print(f"   ✅ {col}: object → string")
        except Exception as e:
            print(f"   ⚠️  {col}: conversion impossible ({e})")
            # En dernier recours, forcer en string basique
            panel_solde_df[col] = panel_solde_df[col].fillna('').astype(str)

    print("\n💾 Export vers S3...")
    object_key = "Solde/out/fusion_1_2021_2025.parquet"
    
    parquet_buffer = io.BytesIO()
    panel_solde_df.to_parquet(parquet_buffer, engine="pyarrow", index=False)
    parquet_buffer.seek(0)

    s3_client.put_object(
        Bucket=bucket_name,
        Key=object_key,
        Body=parquet_buffer.getvalue()
    )
    
    size_mb = len(parquet_buffer.getvalue()) / (1024**2)
    
    print(f"✅ Fichier exporté: s3://{bucket_name}/{object_key}")
    print(f"   • Taille: {size_mb:.2f} MB")
    print(f"   • Lignes: {len(panel_solde_df):,}")
    print(f"   • Colonnes: {panel_solde_df.shape[1]}")

    print("\n✅ PARTIE 1 TERMINÉE AVEC SUCCÈS!")

else:
    print("\n❌ Aucun fichier n'a pu être chargé correctement.")

🚀 PIPELINE SOLDE - PARTIE 1 : FUSION FEUILLES 1

⚙️  Connexion à MinIO/S3...
✅ Connexion établie

📋 Recensement des fichiers Excel...
✅ 58 périodes trouvées

📊 Vérification complétude annuelle:
✅ Année 2021: 12/12 fichiers
✅ Année 2022: 12/12 fichiers
✅ Année 2023: 12/12 fichiers
✅ Année 2024: 12/12 fichiers
⚠️  Année 2025: 10/12 fichiers (manquants: 11, 12)

🔍 Stratégie de détection:
   • Colonnes CRITIQUES (obligatoires): 4
   • Colonnes IMPORTANTES (optionnelles): 22
   • Seuil minimum: 100% critiques + 50% importantes

🔄 Chargement et fusion des fichiers...

✅ 012021.xlsx: 248,266 lignes
✅ 012022.xlsx: 259,833 lignes
✅ 012023.xlsx: 281,226 lignes
✅ 012024.xlsx: 296,935 lignes
✅ 012025.xlsx: 319,736 lignes
✅ 022021.xlsx: 185,336 lignes
✅ 022022.xlsx: 263,019 lignes
✅ 022023.xlsx: 285,415 lignes
✅ 022024.xlsx: 301,043 lignes
✅ 022025.xlsx: 324,441 lignes
✅ 032021.xlsx: 254,964 lignes
✅ 032022.xlsx: 264,498 lignes
✅ 032023.xlsx: 288,060 lignes
✅ 032024.xlsx: 303,825 lignes
✅ 032025.xl

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column GRADE 2 with type object')

In [2]:
# =============================================================================
# STATISTIQUES ET EXPORT
# =============================================================================

print("\n" + "="*80)
print("📊 STATISTIQUES")
print("="*80)
print(f"   • Fichiers traités avec succès  : {stats['succes']}")
print(f"   • Échec détection en-tête       : {stats['echec_header']}")
print(f"   • Échec colonnes manquantes     : {stats['echec_colonnes']}")
print(f"   • Autres erreurs                : {stats['echec_autre']}")
print(f"   • TOTAL                         : {sum(stats.values())}")

if resultat_final:
    print("\n🔗 Concaténation finale...")
    panel_solde_df = pd.concat(resultat_final, ignore_index=True)
    panel_solde_df.dropna(how="all", inplace=True)

    print(f"✅ Panel créé: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")

    print("\n📅 Conversion des dates...")
    colonnes_dates = ["DATE NAISSANCE", "PRISE DE SERVICE", "DATE DEBUT SITUATION", 
                     "DATE RETRAITE", "DATE_COLLECTE"]
    
    for col in colonnes_dates:
        if col in panel_solde_df.columns:
            panel_solde_df[col] = pd.to_datetime(panel_solde_df[col], errors="coerce")

    print("\n🔢 Conversion des valeurs numériques...")
    colonnes_numeriques = ["NOMBRE ENFANT", "MONTANT BRUT", "MONTANT NET",
                          "RETENUE  PENSION", "IMPOT ", "CHARGE PATRONALE",
                          "MOIS/ANNEE", "AGE RETRAITE"]
    
    for col in colonnes_numeriques:
        if col in panel_solde_df.columns:
            panel_solde_df[col] = pd.to_numeric(panel_solde_df[col], errors="coerce")

    # ✅ CORRECTION: FORCER LES COLONNES OBJECT EN STRING
    print("\n🧹 Nettoyage des types pour export Parquet...")
    
    # Identifier toutes les colonnes object
    object_cols = panel_solde_df.select_dtypes(include=['object']).columns
    
    for col in object_cols:
        try:
            # Convertir en string, puis remplacer les valeurs vides par NA
            panel_solde_df[col] = panel_solde_df[col].astype(str)
            panel_solde_df[col] = panel_solde_df[col].replace(['nan', 'None', '<NA>', ''], pd.NA)
            panel_solde_df[col] = panel_solde_df[col].astype("string")
            print(f"   ✅ {col}: object → string")
        except Exception as e:
            print(f"   ⚠️  {col}: conversion impossible ({e})")
            # En dernier recours, forcer en string basique
            panel_solde_df[col] = panel_solde_df[col].fillna('').astype(str)

    print("\n💾 Export vers S3...")
    object_key = "Solde/out/fusion_1_2021_2025.parquet"
    
    parquet_buffer = io.BytesIO()
    panel_solde_df.to_parquet(parquet_buffer, engine="pyarrow", index=False)
    parquet_buffer.seek(0)

    s3_client.put_object(
        Bucket=bucket_name,
        Key=object_key,
        Body=parquet_buffer.getvalue()
    )
    
    size_mb = len(parquet_buffer.getvalue()) / (1024**2)
    
    print(f"✅ Fichier exporté: s3://{bucket_name}/{object_key}")
    print(f"   • Taille: {size_mb:.2f} MB")
    print(f"   • Lignes: {len(panel_solde_df):,}")
    print(f"   • Colonnes: {panel_solde_df.shape[1]}")

    print("\n✅ PARTIE 1 TERMINÉE AVEC SUCCÈS!")

else:
    print("\n❌ Aucun fichier n'a pu être chargé correctement.")


📊 STATISTIQUES
   • Fichiers traités avec succès  : 58
   • Échec détection en-tête       : 0
   • Échec colonnes manquantes     : 0
   • Autres erreurs                : 0
   • TOTAL                         : 58

🔗 Concaténation finale...
✅ Panel créé: 16,778,883 lignes, 30 colonnes

📅 Conversion des dates...

🔢 Conversion des valeurs numériques...

🧹 Nettoyage des types pour export Parquet...
   ✅ MATRICULE||CODE_ORGANISME: object → string
   ✅ MATRICULE: object → string
   ✅ NOM: object → string
   ✅ SEXE: object → string
   ✅ SITUATION MATRIMONIALE: object → string
   ✅ LIEU AFFECTATION: object → string
   ✅ SERVICE: object → string
   ✅ ORGANISME: object → string
   ✅ SITUATION: object → string
   ✅ EMPLOI: object → string
   ✅ FONCTION: object → string
   ✅ GRADE: object → string
   ✅ CLASSE/ECHELON: object → string
   ✅ GRADE 2: object → string
   ✅ PERIODE: object → string
   ✅ SOURCE_FICHIER: object → string

💾 Export vers S3...
✅ Fichier exporté: s3://admindataanstat/Solde/ou

# Associations des codes au libbelés de la fusion des feuilles 1 

In [3]:
# Paramètres
import io
import pandas as pd
import boto3
import pandas as pd
import io
import pandas as pd
import unicodedata
import re
import numpy as np
import unicodedata, re
from pathlib import Path
from pandas.api.types import is_string_dtype
from collections import Counter



# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/out/fusion_1_2021_2025.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))



# Paramètres S3
bucket_name = "admindataanstat"
object_key = "Solde/FICHIER_ANSTAT_CODE_2025.xlsx"  

# Télécharger le fichier depuis S3 dans un buffer mémoire
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(obj['Body'].read())

# Remettre le curseur au début
bytes_io.seek(0)

xlsx = pd.ExcelFile(bytes_io)
print("Feuilles disponibles :", xlsx.sheet_names)



# Je défini une fonction que je nomme "normalize_label"
def normalize_label(s: str) -> str:
    """Minuscules, accents retirés, ponctuation uniformisée, espaces compressés."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""  # uniquement si c'est un vrai NaN
    s = str(s).strip()
    s = (s.replace("-", " ")
           .replace("’", "'")
           .replace("'", " ")
           .replace(".", " "))
    s = s.lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Je défini une fonction que je nomme "_pick_excel_engine" qui a pour objectif d’importer le module openpyxl.
# Si l’import réussit, la fonction renvoie "openpyxl"
# Si openpyxl n’est pas installé, Python va dans ce cas passer au cas exceptionnel. Essayer d’importer xlsxwriter.Si ça marche, il renvoie "xlsxwriter".

def _pick_excel_engine():
    try:
        import openpyxl  # noqa
        return "openpyxl"
    except Exception:
        try:
            import xlsxwriter  # noqa
            return "xlsxwriter"
        except Exception:
            raise RuntimeError("Installez 'openpyxl' ou 'xlsxwriter' pour écrire dans Excel.")

# Je défini une fonction que je nomme "write_sheet" qui a pour rôle d'écrire un DataFrame dans un fichier Excel,
# en remplaçant uniquement la feuille (sheet) spécifiée, sans effacer les autres feuilles du classeur

def write_sheet(df: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit/Remplace la feuille `sheet_name` dans `workbook_path`.
    - Si le classeur n'existe pas: création sans if_sheet_exists.
    - S'il existe: append + replace de la feuille.
    """
    engine = _pick_excel_engine()
    xlsx = Path(workbook_path)

    if engine == "openpyxl":
        if xlsx.exists():
            # append + replace la feuille
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
        else:
            # création (pas de if_sheet_exists en mode 'w')
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="w") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
    else:
        # xlsxwriter: on doit recharger les feuilles existantes pour les réécrire
        sheets = {}
        if xlsx.exists():
            try:
                with pd.ExcelFile(workbook_path) as xf:
                    for sn in xf.sheet_names:
                        sheets[sn] = xf.parse(sn)
            except Exception:
                sheets = {}
        sheets[sheet_name] = df
        with pd.ExcelWriter(workbook_path, engine="xlsxwriter") as xw:
            for sn, d in sheets.items():
                d.to_excel(xw, sheet_name=sn, index=False)

    print(f"✅ Feuille écrite : {sheet_name} → {workbook_path}")


# Définition d'une fonction pour lire et nettoyer les fichiers Excel
def read_reference(
    excel_source,
    sheet_name: str,
    required_cols=None,          # ex: ["CODE_FONCTION","LIBELLÉ_FONCTION"]
    alias_map: dict | None = None, # alias optionnel {col_source: col_cible}, appliqué UNIQUEMENT ici
    drop_title_substr: str | None = "TABLEAU",  # supprime les lignes dont la 1ʳᵉ colonne contient ce motif
    preview_rows: int = 5,
) -> pd.DataFrame:
    """
    Lit une feuille Excel, affiche un aperçu AVANT/APRÈS nettoyage et retourne le DataFrame propre.

    Nettoyage :
      - supprime lignes entièrement vides
      - supprime lignes “titre” si la 1ʳᵉ colonne contient `drop_title_substr` (ex: 'TABLEAU')
      - si besoin, promeut la 1ʳᵉ ligne en entête
      - applique un alias de colonnes (optionnel, UNIQUEMENT au chargement)
      - vérifie la présence des colonnes `required_cols` si fourni
    """

    # --- Lecture brute ---
    df_raw = pd.read_excel(excel_source, sheet_name=sheet_name)

    print(f"\n=== {sheet_name} | APERÇU BRUT (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df_raw.head(preview_rows))
    except Exception:
        print(df_raw.head(preview_rows).to_string(index=False))
    print(f"Colonnes BRUTES ({sheet_name}) :", df_raw.columns.tolist())
    print(f"Shape brut : {df_raw.shape}")

    # --- Copie de travail ---
    df = df_raw.copy()

    # 1) Supprimer les lignes entièrement vides
    df = df.dropna(how="all")

    # 2) Supprimer les lignes de titre si la 1ʳᵉ colonne les contient (ex: 'TABLEAU')
    if drop_title_substr is not None and not df.empty:
        first_col = df.columns[0]
        mask_title = df[first_col].astype(str).str.contains(drop_title_substr, case=False, na=False)
        df = df.loc[~mask_title].copy()

    # 3) Re-détecter l'entête si nécessaire (heuristique simple)
    def _looks_like_header(row_vals, required, alias):
        vals = [str(v).strip() for v in row_vals]
        if required and any(c in vals for c in required):
            return True
        if alias and any(src in vals for src in alias.keys()):
            return True
        if any("unnamed" in v.lower() for v in vals):
            return False
        if any(("code" in v.lower()) or ("lib" in v.lower()) for v in vals):
            return True
        return False

    if not df.empty:
        candidate = df.iloc[0].tolist()
        if _looks_like_header(candidate, required_cols, alias_map):
            df.columns = [str(v).strip() for v in candidate]
            df = df.drop(df.index[0]).reset_index(drop=True)

    # 4) Alias de colonnes (optionnel, seulement au chargement)
    if alias_map:
        df = df.rename(columns=alias_map)

    # 5) Vérifier les colonnes requises (si fourni)
    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print("\n⚠️ Colonnes manquantes après nettoyage/alias :", missing)
            print("Colonnes disponibles :", df.columns.tolist())
            raise ValueError(f"Colonnes manquantes : {missing}")

    # 6) Aperçu après nettoyage
    print(f"\n=== {sheet_name} | APRÈS NETTOYAGE (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df.head(preview_rows))
    except Exception:
        print(df.head(preview_rows).to_string(index=False))
    print(f"Colonnes NETTOYÉES ({sheet_name}) :", df.columns.tolist())
    print(f"Shape nettoyé : {df.shape}")

    return df.reset_index(drop=True)


def check_code_to_label_uniqueness(df_ref: pd.DataFrame, var_code_ext: str, var_lib_ext: str) -> pd.Series:
    """
    Vérifie que chaque code (var_code_ext) pointe vers un seul libellé (var_lib_ext).
    Retourne la série des codes non-uniques (counts).
    """
    verif_code = df_ref.groupby(var_code_ext)[var_lib_ext].nunique()
    bad = verif_code[verif_code > 1]
    if bad.empty:
        print("✅ Chaque code est associé à un seul libellé.")
    else:
        print("❌ Codes associés à plusieurs libellés :")
        print(bad)
    return bad

def ensure_normalized_column(df_ref: pd.DataFrame, var_lib_ext: str, var_norm_ext: str | None = None) -> str:
    """
    Ajoute la colonne normalisée (var_norm_ext) si absente. Retourne son nom.
    """
    var_norm_ext = var_norm_ext or f"{var_lib_ext}_NORM"
    if var_norm_ext not in df_ref.columns:
        df_ref[var_norm_ext] = df_ref[var_lib_ext].apply(normalize_label)
    return var_norm_ext

def detect_normalized_duplicates(df_ref: pd.DataFrame,
                                 var_norm_ext: str,
                                 var_lib_ext: str,
                                 var_code_ext: str) -> pd.DataFrame:
    """
    Détecte les doublons après normalisation et retourne toutes les lignes uniques concernées.

    Paramètres
    ----------
    df_ref : pd.DataFrame
        Table de référence contenant les codes et libellés normalisés.
    var_norm_ext : str
        Nom de la colonne contenant les libellés normalisés.
    var_lib_ext : str
        Nom de la colonne contenant les libellés originaux.
    var_code_ext : str
        Nom de la colonne contenant les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes uniques dont la clé normalisée apparaît plusieurs fois.
    """
    # Compter combien de fois chaque clé normalisée apparaît
    counts = df_ref[var_norm_ext].value_counts()
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code_ext, var_lib_ext, var_norm_ext])

    # Filtrer toutes les lignes concernées
    dup_rows = df_ref[df_ref[var_norm_ext].isin(repeated_keys)].copy()

    # Garder uniquement les lignes uniques (code + libellé + normalisé)
    dup_rows = dup_rows.drop_duplicates(subset=[var_code_ext, var_lib_ext, var_norm_ext])

    # Message d’alerte clair
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes uniques concernées.")

    return dup_rows


def report_duplicates(df: pd.DataFrame, var_norm: str, var_lib: str, var_code: str) -> pd.DataFrame:
    """
    Affiche la liste exhaustive des doublons après normalisation et retourne toutes les lignes concernées.
    
    Paramètres
    ----------
    df : pd.DataFrame
        Table contenant les libellés et codes.
    var_norm : str
        Colonne avec les libellés normalisés (clé de regroupement).
    var_lib : str
        Colonne avec les libellés originaux.
    var_code : str
        Colonne avec les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes du DataFrame où une même clé normalisée apparaît plusieurs fois.
    """

    # --- 1) Compter toutes les occurrences de la clé normalisée ---
    counts = df[var_norm].value_counts()

    # --- 2) Sélectionner les clés répétées ---
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code, var_lib, var_norm])

    # --- 3) Filtrer toutes les lignes concernées ---
    dup_rows = df[df[var_norm].isin(repeated_keys)].copy()

    # --- 4) Message d’alerte ---
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes concernées.")

    # --- 5) Affichage des colonnes essentielles pour l’audit ---
    try:
        from IPython.display import display
        display(dup_rows[[var_code, var_lib, var_norm]])
    except Exception:
        print(dup_rows[[var_code, var_lib, var_norm]].to_string(index=False))

    # --- 6) Retourner le DataFrame complet ---
    return dup_rows



# La fonction "build_random_mapping" a pour but de créer une association unique entre des codes et les libellés normalisés dans un DataFrame,
# en gérant les ambiguïtés et en ajoutant d'autres informations

def build_random_mapping(
    df_ref: pd.DataFrame, # Le DataFrame source contenant les données (codes, libellés brutww, libellés normalisés)
    var_norm_ext: str, # Nom de la colonne "libellé normalisé" sur laquelle on veut créer le mapping
    var_code_ext: str, # Nom de la colonne "code" qui sera associé au libellé normalisé
    var_lib_ext: str, # Nom de la colonne "libellé brut" correspondant au libellé de départ avant traitement
    seed: int = 123 # Seed pour rendre le tirage aléatoire reproductible
) -> pd.DataFrame:
    """
    Construit un mapping 1 code / clé normalisée :
      - choisit aléatoirement un code quand plusieurs existent (traçable par seed),
      - ajoute les colonnes d’audit : EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES.
    Sortie : DataFrame avec colonnes [var_lib_ext, var_norm_ext, var_code_ext,
                                     EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES, CODE_CHOISI]
    """
    df = df_ref.copy() # On crée une copie du DataFrame d’origine
    # infos par clé
    codes_list = (df.groupby(var_norm_ext)[var_code_ext]
                    .apply(lambda s: sorted(map(str, pd.unique(s))))
                    .rename("CODES_LIST")) # créer pour chaque libellé normalisé (var_norm_ext) la liste des codes uniques associés (var_code_ext).
    codes_info = codes_list.to_frame()
    codes_info["NB_CODES_POSSIBLES"] = codes_info["CODES_LIST"].str.len() # Compte le nombre de codes possibles pour chaque libellé normalisé
    codes_info["EST_AMBIGU"] = codes_info["NB_CODES_POSSIBLES"].gt(1) #Crée une colonne booléenne indiquant si le libellé normalisé est associé à plusieurs codes.1" → True si plus d’un code, False sinon
    codes_info["CODES_POSSIBLES"] = codes_info["CODES_LIST"].apply(lambda L: ", ".join(L))

    # tirage aléatoire d'un code par clé normalisée
    pairs_uniques = df[[var_norm_ext, var_code_ext]].drop_duplicates() # On prend seulement les colonnes libellé normalisé (var_norm_ext) et code (var_code_ext) du DataFrame. Ensuite, on supprime les lignes identiques pour ne garder qu’une seule occurrence par combinaison
    selection = (pairs_uniques.groupby(var_norm_ext, group_keys=False)
                 .apply(lambda g: g.sample(n=1, random_state=seed))
                 .reset_index(drop=True)
                 .rename(columns={var_code_ext: "CODE_CHOISI"}))
    # groupby(var_norm_ext, group_keys=False) : On regroupe les lignes par libellé normalisé et chaque groupe contient tous les codes possibles pour ce libellé.
    # .apply(lambda g: g.sample(n=1, random_state=seed)) : pour chaque libellé, on choisit aléatoirement un code parmi les codes possibles. random_state=seed permet que le tirage soit répétable : même code choisi si on réexécute le programme avec le même seed. reset_index(drop=True) : Réinitialise l’index du DataFrame résultant pour avoir un index simple allant de 0 à n-1.
    # .rename(columns={var_code_ext: "CODE_CHOISI"}) : renomme la colonne du code choisi pour clarifier que c’est celui sélectionné aléatoirement.

    # mapping final = libellé brut x clé norm + code choisi + audit d’ambiguïté
    mapping = (df[[var_lib_ext, var_norm_ext]]
               .drop_duplicates() 
               # On récupère les colonnes libellé brut (var_lib_ext) et libellé normalisé (var_norm_ext). Ensuite, on supprime les doublons pour avoir une seule ligne par combinaison libellé brut / libellé normalisé.
               
               .merge(selection, on=var_norm_ext, how="left") 
               # On ajoute la colonne CODE_CHOISI à chaque libellé normalisé. un code choisi aléatoirement par libellé normalisé.
               
               .merge(codes_info.drop(columns=["CODES_LIST"]), left_on=var_norm_ext, right_index=True, how="left"))
                # On ajoute les informations d’audit (nombre de codes possibles, liste des codes, indicateur d’ambiguïté) depuis codes_info. on supprime la liste brute des codes pour ne garder que les colonnes utiles pour l’audit.
    mapping[var_code_ext] = mapping["CODE_CHOISI"]

    cols = [var_lib_ext, var_norm_ext, var_code_ext, "CODE_CHOISI", "EST_AMBIGU",
            "NB_CODES_POSSIBLES", "CODES_POSSIBLES"]
    mapping = mapping[cols].sort_values([ "EST_AMBIGU", var_norm_ext, var_lib_ext ],
                                        ascending=[False, True, True])
    return mapping

def export_mapping(mapping: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit UNIQUEMENT la feuille de mapping (pas de feuille ‘Ambigus’) ;
    la feuille contient les colonnes d’audit ‘EST_AMBIGU’, ‘CODES_POSSIBLES’, etc.
    """
    write_sheet(mapping, workbook_path, sheet_name)



def merge_mapping_into_panel(panel_df: pd.DataFrame,
                             mapping: pd.DataFrame,
                             var_norm_panel: str, var_norm_ext: str,
                             var_code_panel: str, var_code_ext: str,
                             var_lib_panel: str):
    """
    Met à jour les codes du panel à partir du mapping via les libellés normalisés.
    Le code est récupéré directement du mapping et prend le nom de la colonne du panel.
    """
    # 1️⃣ Copie du panel
    df = panel_df.copy()

    # 2️⃣ Créer la colonne normalisée dans le panel si absente
    if var_norm_panel not in df.columns:
        df[var_norm_panel] = df[var_lib_panel].apply(normalize_label)

    # 3️⃣ Préparer le mapping (suppression doublons)
    mapping_clean = mapping[[var_norm_ext, var_code_ext]].drop_duplicates()
    mapping_clean[var_code_ext] = mapping_clean[var_code_ext].astype("string").fillna("NA")

    # 4️⃣ Merge sur la clé normalisée et renommer la colonne du code pour matcher le panel
    out = df.merge(mapping_clean,
                   left_on=var_norm_panel,
                   right_on=var_norm_ext,
                   how="left")\
            .rename(columns={var_code_ext: var_code_panel})

    # 5️⃣ Supprimer les colonnes techniques issues du merge
    out = out.drop(columns=[var_norm_ext], errors="ignore")

    # 6️⃣ Statistiques
    nb_rens = int(out[var_code_panel].notna().sum())
    nb_miss = int(out[var_code_panel].isna().sum())
    print(f"Codes renseignés dans {var_code_panel} : {nb_rens}")
    print(f"Codes manquants dans {var_code_panel} : {nb_miss}")

    return out, nb_rens, nb_miss



#  la fonction "list_unmatched_norms" liste toutes les libellés normalisés (var_norm_panel) dans le panel qui n’ont pas de code associé (var_code_panel vide ou NA).
# Cela permet de détecter les éléments du panel qui n’ont pas pu être appariés avec le mapping.


''' Définition d’une fonction qui prend : 
(1) panel_df  qui est le tableau de données principal (panel)
(2) var_code_panel : la colonne du code qui peut être manquante (NA)
(3) var_norm_panel : la colonne du libellé normalisé '''

def list_unmatched_norms(panel_df: pd.DataFrame, var_code_panel: str, var_norm_panel: str) -> pd.DataFrame:
    """
    Retourne les libellés normalisés non appariés avec le nombre de lignes correspondantes,
    y compris les libellés vides ou entièrement blancs.
    """
    # Sélection des lignes où le code est NA
    s = panel_df.loc[panel_df[var_code_panel].isna(), var_norm_panel].astype(str)

    # Nettoyer les chaînes (supprimer espaces en début/fin)
    s = s.str.strip()

    # Compter les occurrences de chaque libellé, y compris les vides
    counts = s.value_counts(dropna=False).reset_index()
    counts.columns = [var_norm_panel, "COUNT"]

    if counts.empty:
        print("✅ Tous les libellés ont été appariés, aucun non apparié.")
    else:
        total_non_apparies = counts["COUNT"].sum()
        print(f"⚠️ {len(counts)} libellés non appariés trouvés (total lignes concernées : {total_non_apparies}) :")
        try:
            from IPython.display import display
            display(counts)
        except Exception:
            print(counts.to_string(index=False))

    return counts
        
# Cette fonction "best_suggestions" cherche pour chaque libellé non apparié le meilleur candidat dans le référentiel
def best_suggestions(panel_df: pd.DataFrame, mapping: pd.DataFrame,
                     var_norm_panel: str, var_norm_ext: str,
                     var_lib_ext: str, var_code_ext: str,
                     var_code_panel: str,
                     top_n: int = 20,
                     min_score_show: int = 0) -> pd.DataFrame:
    """
    Pour chaque libellé normalisé non apparié dans le panel, calcule le MEILLEUR candidat
    du référentiel (score de similarité). Affiche le top et retourne un DataFrame suggestions_best.

    Paramètres
    ----------
    panel_df : pd.DataFrame
        DataFrame du panel principal (contenant les codes éventuellement manquants)
    mapping : pd.DataFrame
        Référentiel avec les codes et libellés normalisés
    var_norm_panel : str
        Nom de la colonne des libellés normalisés dans le panel
    var_norm_ext : str
        Nom de la colonne des libellés normalisés dans le référentiel
    var_lib_ext : str
        Nom de la colonne des libellés originaux dans le référentiel
    var_code_ext : str
        Nom de la colonne des codes dans le référentiel
    var_code_panel : str
        Nom de la colonne des codes dans le panel
    top_n : int
        Nombre maximum de suggestions à afficher
    min_score_show : int
        Seuil minimal pour afficher les suggestions (score de similarité)
    
    Retour
    ------
    pd.DataFrame
        DataFrame avec les meilleures suggestions pour chaque libellé non apparié.
        Colonnes : [f"{var_norm_panel}_NON_APPARIE", f"MATCH_{var_norm_ext}", "MATCH_SCORE",
                    var_lib_ext, var_code_ext]
    """

    # --- 1) Extraire les libellés non appariés dans le panel ---
    # Utilise list_unmatched_norms qui retourne COUNT, mais ici on prend uniquement la liste
    non_apparies_df = list_unmatched_norms(panel_df, var_code_panel, var_norm_panel)
    non_apparies = non_apparies_df[var_norm_panel].tolist()  # conversion en liste simple

    # --- 2) Préparer la liste des clés normalisées du référentiel ---
    ref_norms = mapping[var_norm_ext].dropna().astype(str).str.strip()
    ref_norms = ref_norms[ref_norms.ne("")].unique().tolist()  # enlever les vides et doublons

    # --- 3) Déterminer le moteur de similarité ---
    use_rapidfuzz = False
    try:
        from rapidfuzz import process, fuzz  # type: ignore
        use_rapidfuzz = True  # RapidFuzz disponible pour calculer la similarité rapidement
    except Exception:
        from difflib import SequenceMatcher  # fallback si RapidFuzz non installé

    # --- 4) Fonction interne pour calculer le meilleur match d’un libellé ---
    def _best_match(q: str):
        if not q or q.strip() == "":
            return None  # pas de texte → pas de match possible
        if use_rapidfuzz:
            # renvoie (libellé normalisé correspondant, score, index)
            return process.extractOne(q, ref_norms, scorer=fuzz.token_set_ratio)
        else:
            # fallback : utiliser SequenceMatcher pour comparer chaque libellé
            best_score, best_idx = -1, None
            for idx, rn in enumerate(ref_norms):
                sc = int(SequenceMatcher(None, q, rn).ratio() * 100)
                if sc > best_score:
                    best_score, best_idx = sc, idx
            return (ref_norms[best_idx], best_score, best_idx) if best_idx is not None else None

    # --- 5) Chercher le meilleur candidat pour chaque libellé non apparié ---
    rows = []
    for qn in non_apparies:
        res = _best_match(qn)

        if res is None:
            # Cas où le libellé est vide ou aucun match trouvé
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": "AUCUNE_SUGGESTION",
                "MATCH_SCORE": 0,
                var_lib_ext: "Aucune suggestion possible",
                var_code_ext: None
            })
            continue

        matched_norm, score, _ = res

        # récupérer la ligne correspondante dans le mapping (libellé et code)
        # si plusieurs correspondances, on prend la première pour la stabilité
        cand = mapping.loc[mapping[var_norm_ext] == matched_norm, [var_lib_ext, var_code_ext]].head(1)
        if cand.empty:
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": matched_norm,
                "MATCH_SCORE": int(score),
                var_lib_ext: "Non trouvé dans mapping",
                var_code_ext: None
            })
            continue

        lib_val = cand.iloc[0][var_lib_ext]
        code_val = cand.iloc[0][var_code_ext]

        # construire une ligne pour le DataFrame des suggestions
        rows.append({
            f"{var_norm_panel}_NON_APPARIE": qn,
            f"MATCH_{var_norm_ext}": matched_norm,
            "MATCH_SCORE": int(score),
            var_lib_ext: lib_val,
            var_code_ext: code_val,
        })

    # --- 6) Colonnes attendues ---
    expected_cols = [
        f"{var_norm_panel}_NON_APPARIE",
        f"MATCH_{var_norm_ext}",
        "MATCH_SCORE",
        var_lib_ext,
        var_code_ext,
    ]

    # --- 7) Création du DataFrame final ---
    suggestions_best = pd.DataFrame(rows)

    # Trier par score puis par libellé pour afficher les meilleures suggestions en haut
    if not suggestions_best.empty:
        suggestions_best = suggestions_best.sort_values(
            ["MATCH_SCORE", var_lib_ext], ascending=[False, True]
        )[expected_cols]
        if min_score_show > 0:
            to_show = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score_show].head(top_n)
        else:
            to_show = suggestions_best.head(top_n)
    else:
        # DataFrame vide structuré avec les colonnes attendues
        suggestions_best = pd.DataFrame(columns=expected_cols)
        to_show = suggestions_best

    # --- 8) Affichage Jupyter-friendly ---
    try:
        from IPython.display import display
        display(to_show)
    except Exception:
        print(to_show.to_string(index=False))

    return suggestions_best


def apply_choices(panel_df: pd.DataFrame,
                  var_code_panel: str, var_norm_panel: str,
                  choices_map: dict[str, str],
                  var_cle_panel: str, var_lib_panel: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Applique un dictionnaire {clé_norm_panel -> code}, uniquement là où le code est NA.
    Renvoie (panel_df_mis_a_jour, audit_df).
    """
    df = panel_df.copy()
    # Masque des lignes où le code est manquant
    fill_mask = df[var_code_panel].isna()

    # Colonne temporaire pour stocker les suggestions de code
    sugg_col = f"{var_code_panel}_SUGG"
    df.loc[fill_mask, sugg_col] = df.loc[fill_mask, var_norm_panel].map(choices_map)

    # Masque des lignes où : var_code_panel est encore NA mais une suggestion a été trouvée dans sugg_col
    m_apply = df[var_code_panel].isna() & df[sugg_col].notna()
    
    # Création d’une colonne BEFORE pour garder une trace de l’état initial des codes avant mise à jour.
    before_col = f"{var_code_panel}_BEFORE" 
    df[before_col] = df[var_code_panel]
    

    # Remplit les codes manquants avec les valeurs suggérées.
    df.loc[m_apply, var_code_panel] = df.loc[m_apply, sugg_col] 
    
    '''
    Crée un DataFrame d’audit qui contient :

           - la clé et le libellé d’origine
           - le code avant mise à jour
           - le code après mise à jour (AFTER)

     Cela permet de suivre exactement quelles lignes ont été modifiées.
     '''

    audit = (df.loc[m_apply, [var_cle_panel, var_lib_panel, var_norm_panel, before_col, var_code_panel]]
               .copy()
               .rename(columns={var_code_panel: f"{var_code_panel}_AFTER"}))

    print(f"✔ Lignes mises à jour : {len(audit)}")
    df.drop(columns=[sugg_col], errors="ignore", inplace=True)
    return df, audit


"""
Automatiser l’application des suggestions de codes sur le panel uniquement pour les suggestions avec un score de similarité supérieur à un seuil min_score.
Elle utilise la fonction apply_choices pour remplir les codes manquants.
Définition de la fonction avec les paramètres :

     - panel_df : le tableau de données principal où on veut remplir les codes.
     - suggestions_best : le DataFrame contenant les suggestions de codes avec score de similarité.
     - var_code_panel : colonne du code dans le panel (celle à remplir).
     - var_norm_panel : colonne du libellé normalisé dans le panel.
     - var_cle_panel : colonne contenant une clé d’identification unique pour chaque ligne.
     - var_lib_panel : colonne du libellé original dans le panel.
     - var_code_ext : colonne du code dans le référentiel de suggestions.
     - min_score : seuil minimal pour appliquer automatiquement la suggestion (défaut 70).

La fonction retourne un tuple (panel_df_mis_a_jour, audit_df).
"""
def auto_apply_by_score(panel_df: pd.DataFrame,
                        suggestions_best: pd.DataFrame,
                        var_code_panel: str, var_norm_panel: str,
                        var_cle_panel: str, var_lib_panel: str,
                        var_code_ext: str,
                        min_score: int = 70) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construit choices_map pour toutes les suggestions avec score ≥ min_score, et applique.
    """
    if suggestions_best.empty: # Vérifie si le DataFrame des suggestions est vide.
        print("INFO : pas de suggestions à appliquer.")
        return panel_df.copy(), pd.DataFrame()  # Si oui : on affiche un message, et on retourne une copie du panel inchangé et un DataFrame d’audit vide.

    # nom de la colonne 'libelle_norm_non_apparie' dans suggestions_best
    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("La table suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # Filtre les suggestions pour ne garder que celles dont le score de similarité est supérieur ou égal à min_score. Seules ces suggestions seront appliquées
    ok = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score]

    # Construit un dictionnaire clé → code à partir des suggestions filtrées : la clé est le libellé normalisé non apparié dans le panel (norm_non_app_col). la valeur est le code correspondant dans le référentiel (var_code_ext).
    # Ce dictionnaire sera utilisé par apply_choices pour remplir automatiquement les codes.
    choices_map = dict(zip(ok[norm_non_app_col], ok[var_code_ext]))

    return apply_choices(panel_df, var_code_panel, var_norm_panel, choices_map, var_cle_panel, var_lib_panel)



def enrich_mapping_with_choices(mapping_final: pd.DataFrame,
                                suggestions_best: pd.DataFrame,
                                var_norm_panel: str,
                                var_norm_ext: str, var_code_ext: str, var_lib_ext: str,
                                choices_map: dict[str, str]) -> pd.DataFrame:
    """
    Ajoute dans le mapping les alias normalisés retenus (clé_norm_panel -> code),
    en remplaçant l’éventuelle ancienne ligne pour cette clé.
    """
    if suggestions_best.empty or not choices_map:
        print("INFO : rien à enrichir (pas de suggestions ou pas de choix).")
        return mapping_final

    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # ne garder que les suggestions dont la clé est choisie
    sel = suggestions_best[suggestions_best[norm_non_app_col].isin(list(choices_map.keys()))].copy()

    # remplacer le code par celui imposé par choices_map (au cas où)
    sel[var_code_ext] = sel[norm_non_app_col].map(choices_map)

    add = (sel[[norm_non_app_col, var_code_ext, var_lib_ext]]
             .drop_duplicates()
             .rename(columns={norm_non_app_col: var_norm_ext}))

    need_cols = {var_norm_ext, var_lib_ext, var_code_ext}
    missing = [c for c in need_cols if c not in mapping_final.columns]
    if missing:
        raise ValueError(f"mapping_final doit contenir {need_cols}, manquant : {missing}")

    before = len(mapping_final)
    # retire les anciennes lignes pour ces clés
    mapping_final = mapping_final[~mapping_final[var_norm_ext].isin(add[var_norm_ext])]
    # concatène + dédoublonne
    mapping_final = (pd.concat([mapping_final, add], ignore_index=True)
                     .drop_duplicates(subset=[var_norm_ext, var_code_ext]))
    print(f"🔁 Mapping enrichi : +{len(mapping_final)-before} lignes (net).")
    return mapping_final


# Cette fonction prend : df : un DataFrame Pandas et cols : une liste de colonnes à vérifier.
# Elle renvoie un DataFrame booléen de la même forme, indiquant si chaque cellule est “renseignée” (c’est-à-dire non vide).

def _filled_mask_for_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Renvoie un DataFrame booléen (mêmes lignes, colonnes=cols) indiquant si chaque cellule est 'renseignée'.
    - Pour colonnes texte: non-NaN ET trim != ""
    - Pour colonnes non-texte: non-NaN
    """
    
    # On crée un sous-DataFrame sub avec uniquement les colonnes à vérifier.
    # --- Sélection des colonnes à vérifier ---
    sub = df[cols]
    filled = sub.notna()  # Tout sauf NaN est considéré comme renseigné. True si la cellule n’est pas NaN, False sinon.

    # Colonnes texte (object, string dtype)
    text_cols = [c for c in cols if c in sub.columns and is_string_dtype(sub[c])]
    if text_cols:
        # Pour ces colonnes: trim et vérifie non vide
        # Pas de .str sur DataFrame -> on applique par colonne (vectorisé)
        trimmed_ne_empty = sub[text_cols].apply(lambda s: s.str.strip().ne(""), axis=0)
        filled[text_cols] = sub[text_cols].notna() & trimmed_ne_empty

    return filled



"""
Cette fonction :

1. Vérifie que toutes les colonnes à analyser existent dans le DataFrame.
2. Utilise _filled_mask_for_cols pour savoir si chaque cellule est renseignée.
3. Compte pour chaque ligne combien de colonnes sont renseignées.
4. Affiche un résumé détaillé du nombre de colonnes renseignées par ligne :
   - Nombre total de lignes
   - Lignes correctes (exactement 1 colonne renseignée)
   - Lignes à problème (0 ou plusieurs colonnes renseignées)
   - Index des lignes problématiques
5. Retourne la liste des lignes qui n’ont pas exactement une seule colonne renseignée,
   ce qui peut poser problème pour les règles de complétude de données.
"""
def check_completeness(df: pd.DataFrame, cols: list[str], title: str) -> list[int]:
    # 1️⃣ Vérification de l’existence des colonnes
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Colonnes manquantes pour [{title}]: {missing}")

    # 2️⃣ Masque booléen des cellules renseignées
    filled = _filled_mask_for_cols(df, cols)
    
    # 3️⃣ Compte combien de colonnes sont renseignées par ligne
    nb_renseignees = filled.sum(axis=1)

    # 4️⃣ Sélection des lignes problématiques (pas exactement 1 colonne renseignée)
    problemes_idx = df.index[nb_renseignees != 1].tolist()

    # 5️⃣ Résumé détaillé
    nb_total = len(df)
    nb_ok = nb_total - len(problemes_idx)
    print(f"\n📊 Résumé de la vérification [{title}] :")
    print(f"🔹 Nombre total de lignes : {nb_total}")
    print(f"✅ Lignes correctes (exactement 1 colonne renseignée) : {nb_ok}")
    print(f"⚠️ Lignes à problème (0 ou plusieurs colonnes renseignées) : {len(problemes_idx)}")
    print(f"📌 Index des lignes problématiques : {problemes_idx}")

    # 6️⃣ Retourne les index des lignes à problème
    return problemes_idx


def drop_if_other_cols_empty(
    df: pd.DataFrame,
    index_to_check: list[int],
    keep_cols: list[str],
    title: str = ""
) -> pd.DataFrame:
    """
    Supprime les lignes dont l'index est dans `index_to_check`
    ET où toutes les colonnes autres que `keep_cols` sont vides (NaN ou "" pour texte).
    """
    # Évite les doublons d'index et garde uniquement ceux qui existent
    idx_check = pd.Index(index_to_check).intersection(df.index)

    # Colonnes à vérifier (toutes sauf keep_cols)
    check_cols = [c for c in df.columns if c not in set(keep_cols)]
    if not check_cols or idx_check.empty:
        print(f"\n🗑️ [{title}] Rien à supprimer (colonnes à vérifier vides ou aucun index valide).")
        return df.reset_index(drop=True)

    # On calcule le masque 'vide' par colonne, SANS convertir tout en string
    sub = df.loc[idx_check, check_cols]

    # Colonnes texte
    text_cols = [c for c in check_cols if is_string_dtype(df[c])]
    empty_text = pd.DataFrame(index=idx_check)
    if text_cols:
        # vide si NaN OU trim == ""
        txt = sub[text_cols]
        empty_text = txt.isna() | txt.apply(lambda s: s.str.strip().eq(""), axis=0)
    else:
        # DataFrame vide mais avec bon index
        empty_text = pd.DataFrame(index=idx_check)

    # Colonnes non-texte: vide si NaN
    non_text_cols = [c for c in check_cols if c not in text_cols]
    empty_nontext = sub[non_text_cols].isna() if non_text_cols else pd.DataFrame(index=idx_check)

    # Concat puis 'all' sur les colonnes pour savoir si toute la ligne est vide côté "autres colonnes"
    if not empty_text.empty and not empty_nontext.empty:
        mask_all_other_empty = pd.concat([empty_text, empty_nontext], axis=1).all(axis=1)
    elif not empty_text.empty:
        mask_all_other_empty = empty_text.all(axis=1)
    elif not empty_nontext.empty:
        mask_all_other_empty = empty_nontext.all(axis=1)
    else:
        # Aucun check_cols (cas déjà géré plus haut, par sécurité)
        mask_all_other_empty = pd.Series(False, index=idx_check)

    final_drop_idx = mask_all_other_empty[mask_all_other_empty].index

    n_all = len(df)
    n_drop = len(final_drop_idx)
    print(f"\n🗑️ [{title}] Suppression de {n_drop} / {n_all} lignes (après vérif des autres colonnes vides)")

    # IMPORTANT: ne copie pas tout le DF inutilement
    return df.drop(index=final_drop_idx).reset_index(drop=True)


import pandas as pd
from pandas.api.types import is_string_dtype

def drop_if_other_cols_empty_ultralight(
    df: pd.DataFrame,
    index_to_check: list[int],
    keep_cols: list[str],
    title: str = "",
    reset_index: bool = False,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Version ultra légère pour grands DataFrames.
    """

    idx_check = pd.Index(index_to_check).intersection(df.index)
    if idx_check.empty:
        if verbose: print(f"\n🗑️ [{title}] Aucun index à vérifier.")
        return df if not reset_index else df.reset_index(drop=True)

    check_cols = [c for c in df.columns if c not in set(keep_cols)]
    if not check_cols:
        if verbose: print(f"\n🗑️ [{title}] Aucune colonne à vérifier.")
        return df if not reset_index else df.reset_index(drop=True)

    sub = df.loc[idx_check, check_cols]

    # Colonnes texte : vide si NaN ou après strip == ""
    mask_text = pd.Series(True, index=sub.index)
    for col in sub.columns:
        if is_string_dtype(df[col]):
            mask_text &= sub[col].isna() | (sub[col].str.strip() == "")

    # Colonnes non-texte : vide si NaN
    mask_nontext = sub.drop(columns=[c for c in sub.columns if is_string_dtype(df[c])], errors="ignore").isna().all(axis=1)

    # Masque final
    mask_all_other_empty = mask_text & mask_nontext
    final_drop_idx = mask_all_other_empty[mask_all_other_empty].index

    df_clean = df.drop(index=final_drop_idx)

    if verbose:
        print(f"\n🗑️ [{title}] Suppression de {len(final_drop_idx)} / {len(df)} lignes.")

    return df_clean.reset_index(drop=True) if reset_index else df_clean


def create_montant_et_matricule(
    df: pd.DataFrame,
    montant_cols: list[str],
    matricule_cols: list[str],
    montant_out: str = "MONTANT",
    matricule_out: str = "MATRICULE_UNIQUE",
    in_place: bool = True,
) -> pd.DataFrame:
    """
    Crée deux colonnes :
      - `montant_out` : première valeur non vide parmi `montant_cols`
                        puis convertie en numérique (NaN si non convertible)
      - `matricule_out`: première valeur non vide parmi `matricule_cols` (après trim)

    Priorité = ordre des listes.
    `in_place=True` modifie le DataFrame d'entrée, sinon retourne une copie.
    """
    def _dedupe(seq):  # conserve l'ordre, supprime doublons
        return list(dict.fromkeys(seq))

    montant_cols = _dedupe([c for c in montant_cols if c in df.columns])
    matricule_cols = _dedupe([c for c in matricule_cols if c in df.columns])

    if not in_place:
        df = df.copy()

    # --- MONTANT ---
    if montant_cols:
        subm = df[montant_cols].copy()

        # Nettoie colonnes texte: trim → NaN si vide
        for c in montant_cols:
            if is_string_dtype(subm[c]) or subm[c].dtype == "object":
                s = subm[c].astype("string").str.strip()
                subm[c] = s.replace("", pd.NA)

        # Prend la 1ère non nulle selon la priorité
        df[montant_out] = subm.bfill(axis=1).iloc[:, 0]

        # Convertit en numérique
        df[montant_out] = pd.to_numeric(df[montant_out], errors="coerce")
    else:
        df[montant_out] = pd.NA  # aucune colonne disponible

    # --- MATRICULE ---
    if matricule_cols:
        subx = df[matricule_cols].copy()

        for c in matricule_cols:
            if is_string_dtype(subx[c]) or subx[c].dtype == "object":
                s = subx[c].astype("string").str.strip()
                subx[c] = s.replace("", pd.NA)
            else:
                # Non-texte (int, float...) → garde tel quel, NaN reste NaN
                pass

        df[matricule_out] = subx.bfill(axis=1).iloc[:, 0]
    else:
        df[matricule_out] = pd.NA

    return df


def create_montant_et_matricule_fast(df, montant_cols, matricule_cols, montant_out="MONTANT", matricule_out="MATRICULE_UNIQUE"):
    # --- Montant ---
    sub_montant = df[montant_cols].astype("string").apply(lambda x: x.str.strip())
    sub_montant = sub_montant.mask(sub_montant == "")
    df[montant_out] = pd.to_numeric(sub_montant.stack().groupby(level=0).first(), errors='coerce')

    # --- Matricule ---
    sub_mat = df[matricule_cols].astype("string").apply(lambda x: x.str.strip())
    sub_mat = sub_mat.mask(sub_mat == "")
    df[matricule_out] = sub_mat.stack().groupby(level=0).first()

    return df



def rename_columns_from_mapping(
    df_panel: pd.DataFrame,
    df_mapping: pd.DataFrame,
    codes_col: str = "CODES_POSSIBLES",
    norm_col: str = "LIBELLÉ_POSTE_NORM",
    make_unique: bool = False,
    sep: str = ','
) -> pd.DataFrame:
    """
    Renomme les colonnes de df_panel selon le mapping codes possibles -> libellé normalisé.
    Signale :
      - les colonnes correspondant à des codes mais sans libellé dans le mapping,
      - les colonnes qui ne sont pas des codes et donc non renommées.
    """
    if norm_col not in df_mapping.columns:
        raise KeyError(f"La colonne {norm_col} est absente du mapping.")

    # Colonnes qui ne sont jamais des codes
    exceptions = ['SOURCE_FICHIER', 'MONTANT BRUT', 'MATRICULE||CODE_ORGANISME']

    # Construire dictionnaire code -> libellé normalisé
    map_code_to_label = {}
    for _, row in df_mapping.iterrows():
        codes_raw = str(row[codes_col]) if pd.notna(row[codes_col]) else ''
        codes_list = [c.strip() for c in codes_raw.split(sep) if c.strip()]
        label = str(row[norm_col]).strip()
        for code in codes_list:
            map_code_to_label[code] = label

    # Renommer les colonnes
    old_cols = [str(c).strip() for c in df_panel.columns]
    renamed_base = [map_code_to_label.get(c, c) for c in old_cols]

    # Gestion collisions si make_unique=True
    if make_unique:
        counts = Counter(renamed_base)
        final_cols = [
            f"{new_lab} (code {orig_code})" if counts[new_lab] > 1 and new_lab != orig_code else new_lab
            for orig_code, new_lab in zip(old_cols, renamed_base)
        ]
    else:
        final_cols = renamed_base

    # Appliquer le renommage
    df_out = df_panel.copy()
    df_out.columns = final_cols

    nb_renamed = sum(oc != nc for oc, nc in zip(old_cols, final_cols))
    print(f"✔ Colonnes renommées via mapping (libellé normalisé) : {nb_renamed}/{len(old_cols)}")

    # Colonnes correspondant à des codes mais sans libellé
    non_renamed_codes = [c_old for c_old, c_new in zip(old_cols, renamed_base)
                         if c_old == c_new and c_old not in exceptions]

    # Colonnes non mapping (exceptions)
    non_mapping_cols = [c for c in old_cols if c in exceptions]

    if non_renamed_codes:
        print(f"⚠️ Colonnes correspondant à des codes mais sans libellé trouvé ({len(non_renamed_codes)}): {non_renamed_codes}")
    else:
        print("✅ Toutes les colonnes correspondant à des codes ont été renommées correctement.")

    if non_mapping_cols:
        print(f"ℹ️ Colonnes non mapping (exclues volontairement) : {non_mapping_cols}")

    return df_out


# Lire la feuille FONCTION dans notre fichier Excel de référence
fichier_ref = "FONCTION"
df_ref = read_reference(bytes_io, fichier_ref)

nm_feuille = "FONCTION" # nom de la feuille Excel à charger
var_code_ext = "CODE_FONCTION" # la colonne de code dans la table externe (fichier excel)
var_lib_ext  = "LIBELLÉ_FONCTION" #la colonne de libellé dans la table externe (fichier Excel)
var_cle_ext  = var_lib_ext # on choisit le libellé comme clé principale côté référence (fichier Excel)

var_cle_panel  = "FONCTION" # clé utilisée dans le fichier panel (base principale)
var_code_panel = "CODE_FONCTION" # colonne de code dans le panel
var_lib_panel  = "FONCTION" # colonne de libellé dans le panel

# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"  # nom de la variable de libellé normalisé dans la table de référence (fichier Excel)
var_norm_panel = f"{var_cle_panel}_NORM"  # nom de la variable de libellé normalisé dans le panel


# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)


# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)


# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)


suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "tresorier charge d etudes g4": "797"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)



#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_FONCTION_BEFORE'] != panel_solde_df['CODE_FONCTION']) | \
            (panel_solde_df['CODE_FONCTION_BEFORE'].isna() & panel_solde_df['CODE_FONCTION'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_FONCTION_BEFORE', 'CODE_FONCTION', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

# Supprime la colonne CODE_FONCTION_BEFORE
panel_solde_df= panel_solde_df.drop(columns=['CODE_FONCTION_BEFORE'])

# Lire la feuille SERVICE dans notre fichier Excel de référence
fichier_ref = "service"
df_ref = read_reference(bytes_io, fichier_ref)

nm_feuille = "SERVICE" # nom de la feuille 
var_code_ext = "CODE_SERVICE"
var_lib_ext  = "LIBELLÉ_SERVICE"
var_cle_ext  = var_lib_ext

var_cle_panel  = "SERVICE"
var_code_panel = "CODE_SERVICE"
var_lib_panel  = "SERVICE"

# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)



# Afficher la liste des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)


# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)


# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "delegation de paris unesco": "5950","ppa urc": "38Q5"
    
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_SERVICE_BEFORE'] != panel_solde_df['CODE_SERVICE']) | \
            (panel_solde_df['CODE_SERVICE_BEFORE'].isna() & panel_solde_df['CODE_SERVICE'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_SERVICE_BEFORE', 'CODE_SERVICE', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)


# Supprimer la variable CODE_SERVICE_BEFORE
panel_solde_df=panel_solde_df.drop(columns=["CODE_SERVICE_BEFORE"])



# Lecture de la feuille ORGANISME du fichier de reférence (Excel)
fichier_ref = "ORGANISME_OK"
df_ref = read_reference(bytes_io, fichier_ref)


"""
Renommer la variable CODE_ORGANISME en ID_ORGANISME pour éviter toutes confusion 
avec la base de données panel où il y a déjà la variable CODE_ORGANISME
"""
df_ref = df_ref.rename({"CODE_ORGANISME": "ID_ORGANISME"}, axis=1)

nm_feuille = "ORGANISME" # nom de la feuille 
var_code_ext = "ID_ORGANISME"
var_lib_ext  = "LIBELLÉ_ORGANISME"
var_cle_ext  = var_lib_ext

var_cle_panel  = "ORGANISME"
var_code_panel = "ID_ORGANISME"
var_lib_panel  = "ORGANISME"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"


# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)


# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)



# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)


# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)


suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)


# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "min des mines, du petrole et de l energie": "14"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)


#  On reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['ID_ORGANISME_BEFORE'] != panel_solde_df['ID_ORGANISME']) | \
            (panel_solde_df['ID_ORGANISME_BEFORE'].isna() & panel_solde_df['ID_ORGANISME'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['ID_ORGANISME_BEFORE', 'ID_ORGANISME', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)


# Supprimer CODE_ORGANISME_BEFORE
panel_solde_df=panel_solde_df.drop(columns=["ID_ORGANISME_BEFORE"])


# Lecture du fichier Excel
fichier_ref = "grade"
df_ref = read_reference(bytes_io, fichier_ref)


nm_feuille = "ECHELONS" # nom de la feuille 
var_code_ext = "CODE_GRADE"
var_lib_ext  = "LIBELLÉ_GRADE"
var_cle_ext  = var_lib_ext

var_cle_panel  = "CLASSE/ECHELON"
var_code_panel = "CODE_GRADE"
var_lib_panel  = "CLASSE/ECHELON"

# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"


# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)


# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)


# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)


# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)


suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "insp general 1er ech": "I1"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)


#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)



# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_GRADE_BEFORE'] != panel_solde_df['CODE_GRADE']) | \
            (panel_solde_df['CODE_GRADE_BEFORE'].isna() & panel_solde_df['CODE_GRADE'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_GRADE_BEFORE', 'CODE_GRADE', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)


# Suppression de la variable CODE_GRADE_BEFORE
panel_solde_df= panel_solde_df.drop(columns=["CODE_GRADE_BEFORE"])


# Renommer la variable pour éviter toute confusion 
panel_solde_df = panel_solde_df.rename(columns={"CODE_GRADE": "CODE_CLASSE/ECHELON"})

# Lecture du fichier Excel
fichier_ref = "HISTORIQUE_ECHELLES_CORPS"
df_ref = read_reference(bytes_io, fichier_ref)


nm_feuille = "EMPLOIS" # nom de la feuille 
var_code_ext = "CODE_EMPLOI"
var_lib_ext  = "LIBELLÉ_EMPLOI"
var_cle_ext  = var_lib_ext

var_cle_panel  = "EMPLOI"
var_code_panel = "CODE_EMPLOI"
var_lib_panel  = "EMPLOI"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"


# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)


# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)


# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)



# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    # ex: "tresorier charge d etudes g4": "797"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)



#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_EMPLOI_BEFORE'] != panel_solde_df['CODE_EMPLOI']) | \
            (panel_solde_df['CODE_EMPLOI_BEFORE'].isna() & panel_solde_df['CODE_EMPLOI'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_EMPLOI_BEFORE', 'CODE_EMPLOI', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)


# Suppression de la variable CODE_EMPLOI_BEFORE
panel_solde_df=panel_solde_df.drop(columns=['CODE_EMPLOI_BEFORE'])





Feuilles disponibles : ['lieu affectation', 'ORGANISME_OK', 'CODE_SITUATION_SOLDE', 'HISTORIQUE_ECHELLES_CORPS', 'ECHELLES_CORPS_ACTUEL', 'service', 'FONCTION', 'grade', 'LIBELLE POSTE']

=== FONCTION | APERÇU BRUT (top 5) ===


,CODE_FONCTION,LIBELLÉ_FONCTION
0,0,
1,1,Personnel Médical Contractuel exerçant des soi...
2,2,Indemnité spécifique au Directeur de Cabinet M...
3,4,Indemnité du Chargé de Mission ou Secrétaire P...
4,5,Moniteur des Sciences Fondamentales


Colonnes BRUTES (FONCTION) : ['CODE_FONCTION', 'LIBELLÉ_FONCTION']
Shape brut : (160, 2)

=== FONCTION | APRÈS NETTOYAGE (top 5) ===


,CODE_FONCTION,LIBELLÉ_FONCTION
0,0,
1,1,Personnel Médical Contractuel exerçant des soi...
2,2,Indemnité spécifique au Directeur de Cabinet M...
3,4,Indemnité du Chargé de Mission ou Secrétaire P...
4,5,Moniteur des Sciences Fondamentales


Colonnes NETTOYÉES (FONCTION) : ['CODE_FONCTION', 'LIBELLÉ_FONCTION']
Shape nettoyé : (160, 2)
✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 5 clés répétées, 11 lignes uniques concernées.
⚠️ Doublons détectés après normalisation : 5 clés répétées, 11 lignes concernées.


...

✅ Feuille écrite : EMPLOIS → Referentiels_unifies.xlsx
Codes renseignés dans CODE_EMPLOI : 16488021
Codes manquants dans CODE_EMPLOI : 290862
⚠️ 2 libellés non appariés trouvés (total lignes concernées : 290862) :


,EMPLOI_NORM,COUNT
0,<na>,268033
1,administrateur des serv finan gen,22829


,EMPLOI_NORM_NON_APPARIE,MATCH_LIBELLÉ_EMPLOI_NORM,MATCH_SCORE,LIBELLÉ_EMPLOI,CODE_EMPLOI
1,administrateur des serv finan gen,administrateur des serv finan finan gen,91,ADMINISTRATEUR DES SERV. FINAN. FINAN. GEN.,9DGH
0,<na>,non classe,28,Non Classé,38ZZ


✔ Lignes mises à jour : 0
INFO : rien à enrichir (pas de suggestions ou pas de choix).
Nombre d'observations modifiées ou complétées : 0


,CODE_EMPLOI_BEFORE,CODE_EMPLOI,EMPLOI_NORM


## Sauvegade de la  base avec les code

In [ ]:
# Connexion S3 / MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"
object_key = "Solde/out/fusion_1_2021_2025_code_new.parquet"

# Sauvegarde dans un buffer mémoire
buffer = io.BytesIO()
panel_solde_df.to_parquet(buffer, engine='pyarrow', index=False)
buffer.seek(0)  # Repositionne au début du buffer

# Upload sur MinIO
s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=buffer)

print(f"✅ Base sauvegardée en Parquet sur MinIO : s3://{bucket_name}/{object_key}")

# Traitement des Differentes variables issues de la fusion des feuilles 1 

In [ ]:
# Paramètres
import io
import pandas as pd
import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re



import pandas as pd
import numpy as np

def build_situation_matrimoniale(
    df: pd.DataFrame,
    source_col: str = "SITUATION MATRIMONIALE",
    out_col: str = "SITUATION_MATRIMONIALE_RECODE",
    mapping: dict | None = None,
    return_report: bool = True
):
    """
    Recoder la situation matrimoniale + tableau effectifs/pourcentages.
    - mapping: dictionnaire de recodage (par défaut celui que tu as donné).
    - out_col: nom de la colonne recodée.
    Retourne: (df, report) si return_report=True, sinon df.
    """
    if mapping is None:
        mapping = {
            "Célibataire": "Célibataire",
            "Cas Particulier": "Autres",
            "Divorcé Séparé": "Divorcé/Séparé",
            "Femme Mariée": "Marié(e)",
            "Invalide Marié": "Marié(e)",
            "Marié": "Marié(e)",
            "Veuf / veuve": "Veuf/Veuve"
        }

    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    # Recodage
    out[out_col] = out[source_col].replace(mapping)

    # Tableaux de synthèse
    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame(
            {"Effectif": eff, "Pourcentage (%)": pct.round(2)}
        ),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return (out, report) if return_report else out


def build_nombre_enfants(
    df: pd.DataFrame,
    source_col: str = "NOMBRE ENFANT",
    out_col: str = "NBRE_ENFTS_IMPUTE",
    fill_value: int | float = 0,
    return_report: bool = True
):
    """
    Créer la variable NBRE_ENFTS_IMPUTE:
    - copie de la variable d'origine,
    - conversion numérique (coercition), NaN -> fill_value (0 par défaut),
    - tableaux effectifs/pourcentages.
    Retourne: (df, report) si return_report=True, sinon df.
    """
    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    # Conversion -> numérique puis imputation des manquants à 0
    out[out_col] = pd.to_numeric(out[source_col], errors="coerce")
    out[out_col] = out[out_col].fillna(fill_value)

    # Si ce sont bien des nombres entiers, cast en Int64 (garde les NA si jamais)
    try:
        out[out_col] = out[out_col].round(0).astype("Int64")
    except Exception:
        # Si des décimaux existent vraiment, on reste en Float64
        out[out_col] = out[out_col].astype("Float64")

    # Tableaux de synthèse
    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame(
            {"Effectif": eff, "Pourcentage (%)": pct.round(2)}
        ),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return (out, report) if return_report else out




def build_categories(
    df: pd.DataFrame,
    grade_col="GRADE",
    grade2_col="GRADE 2",
    compute_categorie2=True,     # calcule CATEGORIE_2 depuis GRADE 2 si la colonne existe
    overwrite_categorie2=True    # réécrit CATEGORIE_2 même si déjà présente
):
    """
    - CATEGORIE_1 depuis GRADE (texte):
        'Non Fonctionnaire' si le texte contient 'non fonctionnaire' (insensible à la casse/espaces)
        sinon extrait la lettre A-D depuis 'Catégorie A7', 'Catégorie B', etc.
        NaN -> 'Non Fonctionnaire'
    - CATEGORIE_2 depuis GRADE 2 (lettres A-D):
        'A7' -> 'A', 'B' -> 'B', autres -> NaN -> 'Non Fonctionnaire'
    - Les 2 colonnes sont typées 'category' avec les mêmes modalités ordonnées.

    Retourne: df (copie) avec CATEGORIE_1 (et CATEGORIE_2 si demandé).
    """

    out = df.copy()
    cat_order = ["Non Fonctionnaire", "A", "B", "C", "D"]

    # ---------- helpers ----------
    def _cat_from_grade2_letter(val):
        """ Cette fonction transforme un GRADE 2 en catégorie """
        if pd.isna(val):
            return pd.NA  # si valeur manquante, on retourne pd.NA.
        s = str(val).strip().upper()  # on convertit en chaîne, on enlève les espaces et on met en majuscules
        if re.fullmatch(r"[ABCD]\d+", s):   # A7, B3, C1, D2...
            return s[0]
        if re.fullmatch(r"[ABCD]", s):      # A, B, C, D
            return s
        return pd.NA

    def _parse_cat_from_grade_text(val):
        if pd.isna(val):
            return pd.NA
        t = str(val)
        # ex.: "Non   fonctionnaire", "non-fonctionnaire"
        if re.search(r"\bnon\s*fonctionnaire\b", t, flags=re.I):
            return "Non Fonctionnaire"
        # ex.: "Catégorie A", "Categorie B7", "catégorie c titulaire"
        m = re.search(r"Cat[ée]gorie\s+([A-D])(?:\s*\d+)?", t, flags=re.I)
        if m:
            return m.group(1).upper()
        return pd.NA

    # ---------- CATEGORIE_1 depuis GRADE ----------
    if grade_col not in out.columns:
        raise KeyError(f"Colonne '{grade_col}' absente.")
    out["CATEGORIE_1"] = out[grade_col].apply(_parse_cat_from_grade_text)
    out["CATEGORIE_1"] = out["CATEGORIE_1"].fillna("Non Fonctionnaire") # Remplace les NaN par "Non Fonctionnaire"
    out["CATEGORIE_1"] = pd.Categorical(out["CATEGORIE_1"], categories=cat_order, ordered=True) #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    # ---------- (optionnel) CATEGORIE_2 depuis GRADE 2 ----------
    if compute_categorie2 and (grade2_col in out.columns):
        if overwrite_categorie2 or ("CATEGORIE_2" not in out.columns):
            cat2_raw = out[grade2_col].apply(_cat_from_grade2_letter)
            cat2 = cat2_raw.astype(object)
            cat2[pd.isna(cat2)] = "Non Fonctionnaire" # Remplace les NaN par "Non Fonctionnaire"
            out["CATEGORIE_2"] = pd.Categorical(cat2, categories=cat_order, ordered=True) # #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    return out 

def build_statut_from_cats(df: pd.DataFrame, 
                           emploi_col: str = "EMPLOI_NORM",
                           cat1_col: str | None = None,
                           cat1_candidates: tuple[str,str] = ("CATEGORIE_1", "CATEGORIE 1"),
                           cat2_col: str | None = None,
                           cat2_candidates: tuple[str,str] = ("CATEGORIE_2", "CATEGORIE 2"),
                           label_cas: str = "Cas particulier"):

    import unicodedata, re
    import numpy as np
    import pandas as pd

    out = df.copy()

    # --- Trouver les colonnes ---
    def _pick_col(cand_list):
        for c in cand_list:
            if c in out.columns:
                return c
        return None

    if cat1_col is None:
        cat1_col = _pick_col(cat1_candidates)
    if cat2_col is None:
        cat2_col = _pick_col(cat2_candidates)

    for col, name in zip([cat1_col, cat2_col, emploi_col], ["CATEGORIE_1","CATEGORIE_2","EMPLOI_NORM"]):
        if col is None or col not in out.columns:
            raise KeyError(f"Colonne '{name}' introuvable.")

    # --- EMPLOI renseigné ---
    emp_norm = out[emploi_col].fillna("").astype(str).str.strip()
    has_emploi = emp_norm.ne("")

    # --- Normalisation catégories ---
    def _norm_ascii_lower(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        s = unicodedata.normalize("NFKD", s)
        s = s.encode("ascii", "ignore").decode("ascii")
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    c1_norm = out[cat1_col].map(_norm_ascii_lower)
    c2_norm = out[cat2_col].map(_norm_ascii_lower)

    c1_is_nf = c1_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)
    c2_is_nf = c2_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)

    c1_is_abcd = c1_norm.str.fullmatch(r"[abcd]", na=False)
    c2_is_abcd = c2_norm.str.fullmatch(r"[abcd]", na=False)

    is_nf_any = c1_is_nf | c2_is_nf
    is_nf_all = c1_is_nf & c2_is_nf
    is_abcd_any = c1_is_abcd | c2_is_abcd
    is_abcd_all = c1_is_abcd & c2_is_abcd

    # --- STATUT ---
    statut = np.select(
        [(has_emploi & is_nf_any) | (~has_emploi & is_abcd_any),
         has_emploi & is_abcd_all,
         ~has_emploi & is_nf_all],
        [label_cas, "Fonctionnaire", "Non Fonctionnaire"],
        default="Non Fonctionnaire"
    ).astype(object)

    out["STATUT"] = pd.Categorical(
        statut,
        categories=["Non Fonctionnaire", "Fonctionnaire", label_cas],
        ordered=True
    )

    # --- Rapport automatique ---
    rep = {
        "repartition_statut": out["STATUT"].value_counts().reindex(
            ["Non Fonctionnaire","Fonctionnaire",label_cas], fill_value=0
        )
    }

    return out, rep



def build_statut_def_periode(
    df: pd.DataFrame,
    statut_col: str = "STATUT",
    matricule_col: str = "MATRICULE",
    periode_col: str = "PERIODE",
    output_col: str = "Statut_def_mois",
    return_report: bool = True
):

    """
        Règle:
      - Pour chaque groupe (MATRICULE et PERIODE ), on définit le statut final:
          * Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
          * Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
          * Sinon → 'Non Fonctionnaire'
          
          Colonnes attendues: statut_col, matricule_col, periode_col
    """
    out = df.copy()

    # Vérifications
    for col in [statut_col, matricule_col, periode_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Normalisation
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Agrégation par groupe. On regroupe par MATRICULE et PERIODE pour vérifier si au moins
    # une ligne est “Fonctionnaire” ou “Cas particulier”.
    key_cols = [matricule_col, periode_col]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Priorité pour le statut final
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        grp_summary = out.groupby(key_cols)[output_col].first()
        rep = {
            "Statut définifitif": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0),
        }
        return out, rep

    return out



def build_statut_def_annee(df, 
                            statut_col="STATUT", 
                            matricule_col="MATRICULE",
                            date_collecte_col="DATE_COLLECTE",
                            output_col="Statut_def_annee",
                            return_report=True):
    """
    Détermine le statut définitif par MATRICULE et ANNEE (extraite de DATE_COLLECTE).
    
    Règle:
      - Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
      - Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
      - Sinon → 'Non Fonctionnaire'
    
    Retour:
        - df mis à jour
        - report (distribution par Statut_def) si return_report=True
    """
    out = df.copy()

    # Vérifications colonnes
    for col in [statut_col, matricule_col, date_collecte_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Extraire l'année
    out[date_collecte_col] = pd.to_datetime(out[date_collecte_col], errors="coerce")
    out["ANNEE"] = out[date_collecte_col].dt.year

    # Normalisation du statut
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Regroupement par matricule et année
    key_cols = [matricule_col, "ANNEE"]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Détermination du statut définitif
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        rep = {
            "Statut_def_distribution": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0)
        }
        return out, rep

    return out



import pandas as pd
import numpy as np

def build_sexe_clean(
    df: pd.DataFrame,
    id_col="MATRICULE",
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    new_col="SEXE_CLEAN"
):
    """
    Crée une variable propre du sexe (SEXE_CLEAN) en prenant, pour chaque individu (id_col),
    la dernière valeur non nulle observée selon la date collect_col.

    Règles:
      - Utilise uniquement les lignes avec date ET sexe non nuls pour déterminer la "dernière" valeur.
      - En cas d’égalité parfaite sur la date, on prend la dernière occurrence dans l’ordre des lignes.
      - Si un individu n’a jamais de sexe non nul (ou pas de date valable), SEXE_CLEAN = NaN.
    """
    df = df.copy()
    # Sécuriser la date
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # Ordre de ligne pour casser les ex æquo au besoin (stable, reproductible)
    df["_row_order"] = np.arange(len(df))

    # Garder uniquement les lignes exploitables pour définir la "dernière" info
    base = (
        df.dropna(subset=[collect_col, sex_col])
          .sort_values([id_col, collect_col, "_row_order"])
    )

    # Prendre la dernière valeur (par date puis par ordre de ligne)
    latest = (
        base.groupby(id_col, as_index=False)
            .last()[[id_col, sex_col]]
            .rename(columns={sex_col: new_col})
    )

    # Fusionner dans le df complet (toutes les lignes récupèrent la même valeur propre par id)
    df = df.drop(columns=[new_col], errors="ignore").merge(latest, on=id_col, how="left")

    # Nettoyage
    df = df.drop(columns=["_row_order"])

    return df


def imputer_sexe(
    df: pd.DataFrame,
    sex_col="SEXE_CLEAN",         # ← on impute désormais à partir de SEXE_CLEAN
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
):
    """
    Impute les valeurs manquantes de `sex_col` en utilisant le mode par (ANNEE_COLLECTE × MOIS_COLLECTE),
    et stocke le résultat dans `sex_valid_col`.
    """
    import pandas as pd
    import numpy as np
    from IPython.display import display

    df = df.copy()
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # Clés temporelles
    df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    df["MOIS_COLLECTE"]  = df[collect_col].dt.month

    # Colonne imputée initialisée avec le propre (SEXE_CLEAN)
    df[sex_valid_col] = df[sex_col]

    # Mode par groupe (année × mois) calculé sur la variable propre
    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # Imputation: si NaN, remplacer par le mode du groupe; sinon, garder la valeur
    df[sex_valid_col] = df.apply(
        lambda row: mode_sexe_groupes.get(
            (row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), row[sex_valid_col]
        ) if pd.isna(row[sex_valid_col]) else row[sex_valid_col],
        axis=1
    )

    # Stats
    tab_sexe_avant = df[sex_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres = df[sex_valid_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres_pct = df[sex_valid_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    crosstab_sexe = pd.crosstab(
        df[sex_col], df[sex_valid_col],
        dropna=False, margins=True, margins_name="Total"
    )

    report = {
        "tab_sexe_avant": tab_sexe_avant,
        "tab_sexe_apres": tab_sexe_apres,
        "tab_sexe_apres_pct": tab_sexe_apres_pct,
        "crosstab_sexe": crosstab_sexe
    }

    if debug:
        print("=== Statistiques avant imputation (SEXE_CLEAN) ===")
        display(tab_sexe_avant.to_frame("Effectif"))
        print("\n=== Statistiques après imputation (SEXE_IMPUTE) ===")
        display(tab_sexe_apres.to_frame("Effectif"))
        print("\n=== Pourcentages après imputation ===")
        display(tab_sexe_apres_pct.to_frame("Pourcentage (%)"))
        print("\n=== Tableau croisé avant/après ===")
        display(crosstab_sexe)

    return df, report




def _to_datetime_mixed(s):
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

def _clean_dates_generic(series):
    s = series.copy()
    s_str = s.astype(str).str.strip().str.lower()
    time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
    zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
    empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])
    s = s.mask(time_only | zero_date | empties, np.nan)
    dt = _to_datetime_mixed(s)
    serial = pd.to_numeric(s_str, errors="coerce")
    serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
    if serial_mask.any():
        dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                   origin="1899-12-30", errors="coerce")
        dt.loc[serial_mask] = dt_serial
    return dt

def _mode_safe(s: pd.Series):
    s_nonan = s.dropna()
    if s_nonan.empty: 
        return pd.NA
    m = s_nonan.mode()
    return m.iloc[0] if not m.empty else pd.NA


# ============================================================
# 1) ÂGE OBSERVÉ / IMPUTÉ / STABILISÉ
# ============================================================
def build_age_valide(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    age_min=18, age_max=70,
    do_impute_age=True,
    stabilize_age=True,
    stabilize_method="last"   # 'last' | 'mode' | 'median'
):
    """
    Construit AGE, AGE_VALIDE, AGE_IMPUTE et AGE_IMPUTE_STABLE (optionnel).
    - Nettoie dates.
    - Pour la naissance, retient la + récente par matricule.
    - AGE précis (anniversaire).
    - Impute AGE par médiane (ANNEE×MOIS×SEXE si dispo).
    - Stabilise AGE_IMPUTE par matricule (dernière/mode/médiane).
    """
    df = df.copy()
    df.columns = pd.Index(df.columns).map(str).str.strip()

    # --- Collecte
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # --- Naissance
    df["DATE_NAISSANCE_CLEAN"] = _clean_dates_generic(df[birth_col])
    # la + récente observée
    tmp = (
        df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
          .sort_values([matricule_col, collect_col], ascending=[True, False])
          .drop_duplicates(subset=[matricule_col], keep="first")
          [[matricule_col, "DATE_NAISSANCE_CLEAN"]]
          .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"})
    )
    df = df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], errors="ignore")
    df = df.merge(tmp, on=matricule_col, how="left", validate="many_to_one")
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")

    # --- AGE précis à la collecte
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref   = df[collect_col]
    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    if mask.any():
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        before_bday = (ref[mask].dt.month < birth[mask].dt.month) | (
            (ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)
        )
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")
    df["AGE"] = age
    df["AGE_VALIDE"] = df["AGE"].where(df["AGE"].between(age_min, age_max))

    # --- AGE_IMPUTE (médiane ANNEE×MOIS×SEXE si dispo)
    if do_impute_age:
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if sex_col in df.columns:
            group_keys.append(sex_col)
        med = df.groupby(group_keys)["AGE_VALIDE"].transform("median")
        global_med = df["AGE_VALIDE"].median()
        df["AGE_IMPUTE"] = (
            df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med.fillna(global_med))
            .round(0).astype("Int64")
        )

    # --- Stabilisation par matricule
    if stabilize_age:
        g = df.sort_values([matricule_col, collect_col]).groupby(matricule_col)["AGE_IMPUTE"]
        if stabilize_method == "last":
            stable = g.last()
        elif stabilize_method == "mode":
            stable = g.apply(_mode_safe)
        elif stabilize_method == "median":
            stable = g.median()
        else:
            raise ValueError("stabilize_method ∈ {'last','mode','median'}")
        df = df.merge(stable.rename("AGE_IMPUTE_STABLE"), on=matricule_col, how="left")

        # Bilan rapide
        nb_multi_age = df.groupby(matricule_col)["AGE_IMPUTE"].nunique(dropna=True).gt(1).sum()
        tot = df[matricule_col].nunique()
        print(f"[AGE] Matricules avec AGE_IMPUTE variable : {nb_multi_age:,} ({nb_multi_age/tot*100:.2f}%)")
        print(f"[AGE] Stabilisation : {stabilize_method}")

    return df




def build_age_service(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    age_min=18, age_max=40,
    days_per_year=365,
    age_src_priority=("AGE_IMPUTE_STABLE","AGE_IMPUTE","AGE"),  # ← ordre de préférence
    stabilize_service=True,
    stabilize_method="mode",  # 'last' | 'mode' | 'median'
    return_tables=True,
    sample_anomalies=10,
    recompute_corrected=True
):
    """
    Construit l'âge au service et le stabilise.

    Sorties principales:
      - PRISE_DE_SERVICE_CLEAN, PRISE_DE_SERVICE_CORRIGEE
      - AGE_AU_SERVICE, AGE_AU_SERVICE_VALIDE, AGE_AU_SERVICE_IMPUTE
      - FLAG_MULTI_AGE_SERVICE
      - AGE_AU_SERVICE_FINAL (si stabilize_service=True)
    Logique:
      1) Nettoyage PDS + correction = valeur observée la plus récente par matricule (si recompute_corrected=True)
      2) Choix de la source d'âge selon age_src_priority (première colonne existante)
      3) AGE_AU_SERVICE = âge_source - floor((collecte - PDS_corrigée)/365)
      4) Bornage [age_min; age_max] → AGE_AU_SERVICE_VALIDE
      5) Imputation des manquants par médiane (ANNEE×MOIS×SEXE si dispo) sinon médiane globale
      6) Stabilisation par matricule (mode/last/median) → AGE_AU_SERVICE_FINAL
    """
    df = df.copy()
    df.columns = pd.Index(df.columns).map(str).str.strip()

    # --- 0) DATE_COLLECTE & dérivées
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if year_collect_col not in df.columns:
        df[year_collect_col] = df[collect_col].dt.year
    if month_collect_col not in df.columns:
        df[month_collect_col] = df[collect_col].dt.month

    # --- 1) PRISE_DE_SERVICE_CLEAN
    df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df[start_col_raw])

    # --- 2) Diagnostics PDS multiples
    nb_dates_par_matricule = df.groupby(matricule_col)["PRISE_DE_SERVICE_CLEAN"].nunique(dropna=True)
    matricules_problemes = nb_dates_par_matricule[nb_dates_par_matricule > 1].index.tolist()
    anomalies_dates = (
        df[df[matricule_col].isin(matricules_problemes)]
        [[matricule_col, collect_col, "PRISE_DE_SERVICE_CLEAN"]]
        .sort_values([matricule_col, collect_col])
    )

    # --- 3) PRISE_DE_SERVICE_CORRIGEE = valeur la + récente observée
    if recompute_corrected or (start_col_corrected not in df.columns):
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (
            panel_sorted.drop_duplicates(subset=[matricule_col], keep="first")
            [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
            .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected})
        )
        df = df.drop(columns=[start_col_corrected], errors="ignore")
        df = df.merge(ps_corr, on=matricule_col, how="left", validate="many_to_one")

    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])

    # --- 4) Colonnes de sortie (âge au service)
    df["AGE_AU_SERVICE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
    df["AGE_AU_SERVICE_VALIDE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")

    # --- 5) Choix de la source d'âge (priorité)
    age_src = None
    for col in age_src_priority:
        if col in df.columns:
            age_src = col
            break
    if age_src is None:
        age_src = "_AGE_SRC_TMP_"
        df[age_src] = pd.Series(pd.NA, index=df.index, dtype="Float64")

    df[age_src] = pd.to_numeric(df[age_src], errors="coerce").astype("Float64")

    # --- 6) Calcul AGE_AU_SERVICE quand âge & PDS sont présents
    mask = df[age_src].notna() & df[start_col_corrected].notna()
    if mask.any():
        delta_days = (df.loc[mask, collect_col] - df.loc[mask, start_col_corrected]).dt.days
        df.loc[mask, "AGE_AU_SERVICE"] = (
            df.loc[mask, age_src] - (delta_days // days_per_year)
        ).astype("Float64")

    # --- 7) Bornage de l'âge au service
    ok = df["AGE_AU_SERVICE"].between(age_min, age_max)
    df.loc[ok, "AGE_AU_SERVICE_VALIDE"] = df.loc[ok, "AGE_AU_SERVICE"].astype("Float64")

    # --- 8) Imputation (ANNEE×MOIS×SEXE si dispo ; sinon médiane globale)
    candidate_keys = [year_collect_col, month_collect_col, sex_col]
    group_keys = [k for k in candidate_keys if (k is not None and k in df.columns)]
    global_med = df["AGE_AU_SERVICE_VALIDE"].median()
    if len(group_keys) > 0:
        med_group = df.groupby(group_keys)["AGE_AU_SERVICE_VALIDE"].transform("median")
    else:
        med_group = pd.Series(global_med, index=df.index, dtype="Float64")

    impute_base = df["AGE_AU_SERVICE_VALIDE"].where(
        df["AGE_AU_SERVICE_VALIDE"].notna(), med_group.fillna(global_med)
    )
    df["AGE_AU_SERVICE_IMPUTE"] = pd.to_numeric(impute_base, errors="coerce").round(0).astype("Float64")

    # --- 9) Drapeau multi-âges imputés (avant stabilisation)
    flag_multi = df.groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"].nunique(dropna=True) > 1
    df["FLAG_MULTI_AGE_SERVICE"] = df[matricule_col].map(flag_multi).astype("Int64")

    # --- 10) Stabilisation finale par matricule
    if stabilize_service:
        g = (
            df.sort_values([matricule_col, collect_col])
              .groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"]
        )
        try:
            if stabilize_method == "last":
                final = g.last()
            elif stabilize_method == "mode":
                final = g.apply(_mode_safe)
            elif stabilize_method == "median":
                final = g.median()
            else:
                raise ValueError("stabilize_method ∈ {'last','mode','median'}")
            df = df.merge(final.rename("AGE_AU_SERVICE_FINAL"), on=matricule_col, how="left")
        except Exception:
            # Filet de sécurité : retomber sur l'imputé
            df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]
    else:
        df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]

    # >>> Garde supplémentaire : garantir l'existence de la colonne <<<
    if "AGE_AU_SERVICE_FINAL" not in df.columns:
        df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]

    # --- 11) Rapport (optionnel)
    if return_tables:
        tot = df[matricule_col].nunique()
        before = df.groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"].nunique(dropna=True).gt(1).sum()
        after  = df.groupby(matricule_col)["AGE_AU_SERVICE_FINAL"].nunique(dropna=True).gt(1).sum()
        pct_b = round(before / tot * 100, 2)
        pct_a = round(after / tot * 100, 2)

        print(f"[SERVICE] Multi-âges imputés AVANT stabilisation : {before:,} / {tot:,} ({pct_b}%)")
        print(f"[SERVICE] Multi-âges APRÈS  stabilisation : {after:,} / {tot:,} ({pct_a}%)  → méthode={stabilize_method}")

        tab_valide = df["AGE_AU_SERVICE_VALIDE"].value_counts(dropna=False).sort_index()
        tab_valide_pct = df["AGE_AU_SERVICE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()
        tab_impute = df["AGE_AU_SERVICE_IMPUTE"].value_counts(dropna=False).sort_index()
        tab_impute_pct = df["AGE_AU_SERVICE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

        report = {
            "n_matricules_multi_service_dates": len(matricules_problemes),
            "anomalies_pds_head": anomalies_dates.head(sample_anomalies),
            "tab_age_service_valide": pd.DataFrame({
                "Effectif": tab_valide,
                "Pourcentage (%)": (tab_valide_pct * 100).round(2)
            }),
            "tab_age_service_impute": pd.DataFrame({
                "Effectif": tab_impute,
                "Pourcentage (%)": (tab_impute_pct * 100).round(2)
            }),
            "multi_before_after": {"before": before, "after": after, "total": tot},
            "age_source_used": age_src
        }
        return df, report

    return df



def build_anciennete_mensuelle(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str | None = "CODE_ORGANISME",   # mettre None pour ignorer l’organisme
    collect_col: str = "DATE_COLLECTE",
    start_service_col: str = "PRISE_DE_SERVICE_CORRIGEE",  # ne change pas d’une année à l’autre

    # Fenêtre d’observation (ex. 2015–2020)
    min_period: str = "2015-01",   # inclus
    max_period: str = "2020-12",   # inclus

    # Options de sortie
    return_by_org: bool = True,    # ancienneté par organisme si org_col != None
    return_by_person: bool = True, # ancienneté globale par personne (tous organismes)
    keep_counts: bool = True,      # garder NB_APPA_MOIS_* et PRESENCE_MOIS_*

    # Pré-compte avant la fenêtre (si prise de service antérieure à min_period)
    include_prewindow_base: bool = True
):
    """
    Calcule l'ancienneté en MOIS :
      - avant la fenêtre : si include_prewindow_base=True et prise de service < min_period,
        on ajoute une base d'ancienneté continue du mois de prise de service jusqu'au mois précédent min_period.
      - dans la fenêtre [min_period, max_period] : on additionne UNIQUEMENT les mois effectivement observés
        (présence = au moins une ligne dans le mois).

    Sorties dans le DataFrame retourné (jointes au niveau mensuel) :
      - Par PERSONNE : NB_APPA_MOIS_PERS, PRESENCE_MOIS_PERS, ANCIENNETE_MOIS_PERS,
        BASE_PRE2015_MOIS_PERS (si include_prewindow_base),
        ANCIENNETE_MOIS_PERS_TOTAL (= base pré-fenêtre + observé),
        ANCIENNETE_AN_PERS, ANCIENNETE_AN_PERS_TOTAL.
      - Par ORGANISME : NB_APPA_MOIS_ORG, PRESENCE_MOIS_ORG, ANCIENNETE_MOIS_ORG, ANCIENNETE_AN_ORG.
    """
    out = df.copy()

    # ---------- 0) Dates et borne de fenêtre ----------
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = début de mois (clé mensuelle)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # Fenêtre [min_period, max_period] (début de mois)
    min_p = pd.Period(min_period, freq="M").to_timestamp(how="start")
    max_p = pd.Period(max_period, freq="M").to_timestamp(how="start")

    # Filtre fenêtre
    out = out.loc[(out["PERIODE"] >= min_p) & (out["PERIODE"] <= max_p)].copy()

    # Calendrier mensuel complet pour la fenêtre
    full_months = pd.date_range(min_p, max_p, freq="MS")  # MonthStart (DatetimeIndex)

    # ---------- 1) Prise de service fixe (une par matricule) ----------
    if start_service_col in out.columns:
        tmp_ps = (
            out[[matricule_col, start_service_col]]
            .dropna()
            .sort_values([matricule_col, start_service_col])  # la plus ANCIENNE
            .drop_duplicates(subset=[matricule_col], keep="first")
            .rename(columns={start_service_col: "PRISE_DE_SERVICE_FIXE"})
        )
        out = out.drop(columns=["PRISE_DE_SERVICE_FIXE"], errors="ignore")
        out = out.merge(tmp_ps, on=matricule_col, how="left")
    else:
        out["PRISE_DE_SERVICE_FIXE"] = pd.NaT

    out["PRISE_DE_SERVICE_FIXE"] = pd.to_datetime(out["PRISE_DE_SERVICE_FIXE"], errors="coerce")
    out["START_AVANT_FENETRE"] = out["PRISE_DE_SERVICE_FIXE"].lt(min_p)

    # ---------- 2) Base pré-fenêtre par PERSONNE ----------
    if include_prewindow_base:
        from pandas.tseries.offsets import MonthBegin

        start_window = min_p                      # 1er jour de la fenêtre (ex : 2015-01-01)
        end_pre = start_window - MonthBegin(1)    # 1er jour du mois précédent (ex : 2014-12-01)

        def months_diff_inclusive(d0: pd.Timestamp, d1: pd.Timestamp) -> int:
            """Nb de mois entre d0 et d1, inclusivement, en ignorant les jours."""
            if pd.isna(d0) or pd.isna(d1) or d0 > d1:
                return 0
            return (d1.year - d0.year) * 12 + (d1.month - d0.month) + 1

        base_pre = (
            out[[matricule_col, "PRISE_DE_SERVICE_FIXE"]]
            .dropna(subset=["PRISE_DE_SERVICE_FIXE"])
            .drop_duplicates(subset=[matricule_col], keep="first")
        ).copy()

        base_pre["BASE_PRE2015_MOIS_PERS"] = base_pre["PRISE_DE_SERVICE_FIXE"].apply(
            lambda d: months_diff_inclusive(d, end_pre) if d < start_window else 0
        )

        out = out.merge(
            base_pre[[matricule_col, "BASE_PRE2015_MOIS_PERS"]],
            on=matricule_col, how="left"
        )
        out["BASE_PRE2015_MOIS_PERS"] = out["BASE_PRE2015_MOIS_PERS"].fillna(0).astype(int)
    else:
        out["BASE_PRE2015_MOIS_PERS"] = 0

    # ----------------------------------------------------------------
    # OUTIL : calendrier complet via produit cartésien (évite la perte des colonnes de clé)
    # ----------------------------------------------------------------
    def _anciennete_mensuelle(group_keys_prefix: list[str], suffix: str):
        """
        Construit un tableau mensuel complet par (group_keys_prefix) x PERIODE :
          - NB_APPA_MOIS{suffix} = nombre de lignes dans le mois
          - PRESENCE_MOIS{suffix} = 1 si NB_APPA_MOIS > 0, sinon 0
          - ANCIENNETE_MOIS{suffix} = somme cumulée des PRESENCE_MOIS par groupe
          - ANCIENNETE_AN{suffix} = ANCIENNETE_MOIS{suffix} // 12
        """
        keys = group_keys_prefix + ["PERIODE"]

        # a) Compter les apparitions (toutes lignes) par mois
        counts = (
            out.groupby(keys, dropna=False)
               .size()
               .rename(f"NB_APPA_MOIS{suffix}")
               .reset_index()
        )

        # b) Produit cartésien (groupes × full_months)
        groups = counts[group_keys_prefix].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        groups["_key"] = 1
        months_df["_key"] = 1
        grid = groups.merge(months_df, on="_key").drop(columns="_key")

        # c) Joindre les comptes et compléter à 0
        filled = grid.merge(counts, on=keys, how="left")
        nb_col = f"NB_APPA_MOIS{suffix}"
        filled[nb_col] = filled[nb_col].fillna(0).astype(int)

        # d) Présence binaire
        pres_col = f"PRESENCE_MOIS{suffix}"
        filled[pres_col] = (filled[nb_col] > 0).astype(int)

        # e) Ancienneté cumulée en mois (dans la fenêtre)
        anc_col = f"ANCIENNETE_MOIS{suffix}"
        filled[anc_col] = (
            filled.sort_values(group_keys_prefix + ["PERIODE"])
                  .groupby(group_keys_prefix, dropna=False)[pres_col]
                  .cumsum()
                  .astype("Int64")
        )

        # f) Années (entières)
        anc_year_col = f"ANCIENNETE_AN{suffix}"
        filled[anc_year_col] = (filled[anc_col] // 12).astype("Int64")

        return filled

    # ---------- 3) Ancienneté par ORGANISME ----------
    df_org = None
    if return_by_org and (org_col is not None) and (org_col in out.columns):
        df_org = _anciennete_mensuelle([matricule_col, org_col], "_ORG")

    # ---------- 4) Ancienneté par PERSONNE ----------
    df_pers = None
    if return_by_person:
        # Construire les comptes à partir de out (tous organismes confondus)
        # → On part de out mais on regroupe seulement par matricule et PERIODE.
        tmp = out[[matricule_col, "PERIODE"]].copy()
        tmp["_ones"] = 1
        counts_pers = (
            tmp.groupby([matricule_col, "PERIODE"], dropna=False)["_ones"]
               .sum()
               .rename("NB_APPA_MOIS_PERS")
               .reset_index()
        )

        # Produit cartésien (matricules × full_months)
        pers = counts_pers[[matricule_col]].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        pers["_key"] = 1
        months_df["_key"] = 1
        grid_pers = pers.merge(months_df, on="_key").drop(columns="_key")

        df_pers = grid_pers.merge(counts_pers, on=[matricule_col, "PERIODE"], how="left")
        df_pers["NB_APPA_MOIS_PERS"] = df_pers["NB_APPA_MOIS_PERS"].fillna(0).astype(int)

        df_pers["PRESENCE_MOIS_PERS"] = (df_pers["NB_APPA_MOIS_PERS"] > 0).astype(int)
        df_pers["ANCIENNETE_MOIS_PERS"] = (
            df_pers.sort_values([matricule_col, "PERIODE"])
                   .groupby([matricule_col], dropna=False)["PRESENCE_MOIS_PERS"]
                   .cumsum()
                   .astype("Int64")
        )
        df_pers["ANCIENNETE_AN_PERS"] = (df_pers["ANCIENNETE_MOIS_PERS"] // 12).astype("Int64")

    # ---------- 5) Joins sur le niveau mensuel ----------
    base_keys = [matricule_col, "PERIODE"]

    if (return_by_org and (df_org is not None)):
        out = out.merge(df_org, on=[matricule_col, org_col, "PERIODE"], how="left")

    if (return_by_person and (df_pers is not None)):
        out = out.merge(df_pers, on=base_keys, how="left")

        # Ajouter la base pré-fenêtre à l'ancienneté cumulée PERSONNE
        out["ANCIENNETE_MOIS_PERS_TOTAL"] = (
            out["BASE_PRE2015_MOIS_PERS"].astype(int) + out["ANCIENNETE_MOIS_PERS"].fillna(0).astype(int)
        ).astype("Int64")
        out["ANCIENNETE_AN_PERS_TOTAL"] = (out["ANCIENNETE_MOIS_PERS_TOTAL"] // 12).astype("Int64")

    # ---------- 6) Nettoyage colonnes optionnel ----------
    if not keep_counts:
        drop_cols = [c for c in out.columns if c.startswith("NB_APPA_MOIS")]
        out.drop(columns=drop_cols, inplace=True, errors="ignore")

    # ---------- 7) Report synthétique ----------
    rep = {
        "periode_min": min_p.date(),
        "periode_max": max_p.date(),
        "nb_matricules": out[matricule_col].nunique(),
        "exemple_pers": None
    }
    if return_by_person and ("ANCIENNETE_MOIS_PERS" in out.columns):
        rep["exemple_pers"] = (
            out[[matricule_col, "PERIODE", "NB_APPA_MOIS_PERS", "PRESENCE_MOIS_PERS",
                 "ANCIENNETE_MOIS_PERS", "BASE_PRE2015_MOIS_PERS", "ANCIENNETE_MOIS_PERS_TOTAL"]]
            .sort_values([matricule_col, "PERIODE"])
            .head(20)
        )

    return out, rep



import pandas as pd
import numpy as np

def build_anciennete_organisme_depuis_ps_min_collecte(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str = "ID_ORGANISME",
    collect_col: str = "DATE_COLLECTE",
    ps_col: str = "PRISE DE SERVICE",       # la colonne fournie dans ta base
    entry_col: str = "DATE_ENTREE_ORG",      # date d'entrée retenue (PS à min DATE_COLLECTE)
    months_col: str = "ANCIENNETE_ORG_MOIS",
    years_col: str = "ANCIENNETE_ORG_AN",
    inclusive: bool = True,                  # True => le mois d’entrée compte (mai→mai = 1)
    fallback_when_ps_missing: str = "min_collecte",  
    # "min_collecte" => si la PS est manquante sur la ligne la + vieille, on prend min(DATE_COLLECTE)
    # "min_ps_non_null" => sinon chercher la + vieille PRISE DE SERVICE non nulle du couple
    return_report: bool = True
):
    """
    Logique :
      1) Trouver, pour chaque couple (MATRICULE, ORGANISME), la ligne ayant la plus vieille DATE_COLLECTE.
      2) Prendre la PRISE DE SERVICE de CETTE ligne comme date d'entrée (DATE_ENTREE_ORG).
         - Si cette PS est manquante :
             * fallback_when_ps_missing == "min_ps_non_null" : on prend la plus vieille PS non nulle du couple.
             * sinon (par défaut) : on prend la plus vieille DATE_COLLECTE comme entrée.
      3) Pour chaque ligne, calculer ANCIENNETE_ORG_MOIS = écart calendaire en mois entre DATE_ENTREE_ORG et le mois de la ligne
         (mois d’entrée compté si inclusive=True).
      4) ANCIENNETE_ORG_AN = ANCIENNETE_ORG_MOIS // 12

    Remarque :
      - On n’invente aucune ligne : calcul au niveau des lignes existantes.
    """
    out = df.copy()

    # -- sécuriser dates --
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    out[ps_col] = pd.to_datetime(out[ps_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = mois civil (début de mois)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # 1) identifier, par couple, la ligne à min DATE_COLLECTE
    base = (
        out[[matricule_col, org_col, collect_col, ps_col]]
        .sort_values([matricule_col, org_col, collect_col], ascending=[True, True, True])
        .groupby([matricule_col, org_col], dropna=False, as_index=False)
        .first()   # conserve la ligne à plus vieille collecte
        .rename(columns={ps_col: "_PS_SUR_MINCOLLECTE", collect_col: "_MIN_COLLECTE"})
    )

    # 2) fallback si PS manquante sur la ligne min_collecte
    if fallback_when_ps_missing == "min_ps_non_null":
        # chercher la plus vieille PS non nulle du couple
        min_ps = (
            out[[matricule_col, org_col, ps_col]]
            .dropna(subset=[ps_col])
            .sort_values([matricule_col, org_col, ps_col])
            .groupby([matricule_col, org_col], dropna=False, as_index=False)
            .first()
            .rename(columns={ps_col: "_MIN_PS_NON_NULL"})
        )
        base = base.merge(min_ps, on=[matricule_col, org_col], how="left")

        # règle : PS = _PS_SUR_MINCOLLECTE si dispo, sinon _MIN_PS_NON_NULL, sinon _MIN_COLLECTE
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            np.where(base["_MIN_PS_NON_NULL"].notna(), base["_MIN_PS_NON_NULL"], base["_MIN_COLLECTE"])
        )
    else:
        # règle par défaut : PS = _PS_SUR_MINCOLLECTE si dispo, sinon min collecte
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            base["_MIN_COLLECTE"]
        )

    base = base[[matricule_col, org_col, entry_col]]

    # 3) joindre l'entrée au panel
    out = out.drop(columns=[entry_col], errors="ignore")
    out = out.merge(base, on=[matricule_col, org_col], how="left")
    out[entry_col] = pd.to_datetime(out[entry_col], errors="coerce")

    # 4) calcul des mois inclusifs
    def _months_diff_inclusive(d0, d1, inclusive=True):
        if pd.isna(d0) or pd.isna(d1):
            return np.nan
        m0 = pd.Timestamp(d0).to_period("M")
        m1 = pd.Timestamp(d1).to_period("M")
        if m1 < m0:
            return 0
        base = (m1.year - m0.year) * 12 + (m1.month - m0.month)
        return base + 1 if inclusive else base

    out[months_col] = [
        _months_diff_inclusive(de, pe, inclusive) for de, pe in zip(out[entry_col], out["PERIODE"])
    ]
    out[months_col] = pd.to_numeric(out[months_col], errors="coerce").astype("Int64")

    # 5) années
    out[years_col] = (out[months_col] // 12).astype("Int64")

    if return_report:
        rep = {
            "nb_couples_personne_organisme": int(base.shape[0]),
            "extrait": out[[matricule_col, org_col, "PERIODE", entry_col, months_col, years_col]]
                        .sort_values([matricule_col, org_col, "PERIODE"])
                        .head(20)
        }
        return out, rep

    return out


## Chargement de la base 

In [ ]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"

# object_key 
object_key = "Solde/panel_solde_df_1_code.parquet"  # Chemin dans le bucket

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

In [ ]:
import pandas as pd
import numpy as np

df = panel_solde_df.copy()

# 1) Parser PERIODE (int YYYYMM) -> Period[M]
p_str = pd.to_numeric(df["PERIODE"], errors="coerce").astype("Int64").astype(str).str.zfill(6)
year_p = pd.to_numeric(p_str.str[:4], errors="coerce")
month_p = pd.to_numeric(p_str.str[4:6], errors="coerce")
valid_p = month_p.between(1, 12)
periode_M = pd.Series(pd.NaT, index=df.index, dtype="period[M]")
periode_M[valid_p] = pd.PeriodIndex(year=year_p[valid_p], month=month_p[valid_p], freq="M")

# 2) Parser DATE_COLLECTE (str) -> datetime -> Period[M]
dc = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce") \
       .fillna(pd.to_datetime(df["DATE_COLLECTE"], errors="coerce", dayfirst=True))
date_M = dc.dt.to_period("M")

df["_PERIODE_M"] = periode_M
df["_DATE_COLLECTE_M"] = date_M

# 3) Comparaison au mois
same_month = df["_PERIODE_M"].eq(df["_DATE_COLLECTE_M"])
both_na = df["_PERIODE_M"].isna() & df["_DATE_COLLECTE_M"].isna()

print("=== Résumé ===")
print(f"Total: {len(df):,}")
print(f"Égalité mois: {same_month.sum():,} ({same_month.mean():.1%})")
print(f"Incohérences: {(~same_month & ~both_na).sum():,} ({(~same_month & ~both_na).mean():.1%})")

# 4) Écart en MOIS sans TypeError (méthode année*12 + mois)
valid = df["_PERIODE_M"].notna() & df["_DATE_COLLECTE_M"].notna()

month_diff = pd.Series(pd.NA, index=df.index, dtype="Int64")
y_p = df.loc[valid, "_PERIODE_M"].dt.year
m_p = df.loc[valid, "_PERIODE_M"].dt.month
y_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.year
m_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.month

month_diff.loc[valid] = (y_d*12 + m_d) - (y_p*12 + m_p)
df["_ECART_MOIS"] = month_diff

print("\n=== Distribution des écarts (mois) ===")
print(month_diff.value_counts(dropna=True).sort_index())

# 5) Écart en jours (DATE_COLLECTE - 1er jour de PERIODE), idem avec masque
ecart_jours = pd.Series(pd.NA, index=df.index, dtype="Int64")
periode_as_date = df["_PERIODE_M"].dt.to_timestamp(how="start")
ecart_jours.loc[valid] = (dc.loc[valid] - periode_as_date.loc[valid]).dt.days.astype("Int64")
df["_ECART_JOURS"] = ecart_jours

print("\n=== Écart en jours (stats) ===")
print(ecart_jours.dropna().describe())

# 6) Exemples d’incohérences
mismatch = df.loc[~same_month & ~both_na, ["PERIODE", "DATE_COLLECTE", "_PERIODE_M", "_DATE_COLLECTE_M", "_ECART_MOIS", "_ECART_JOURS"]]
print("\n=== Exemples d'incohérences (5) ===")
print(mismatch.head(5))


In [ ]:
panel_solde_df, rep_sitmat = build_situation_matrimoniale(panel_solde_df)
# Aperçu du tableau (effectifs + %)
rep_sitmat["table"]


panel_solde_df, rep_enfants = build_nombre_enfants(panel_solde_df)
# Aperçu du tableau (effectifs + %)
rep_enfants["table"]

# Appliquer la fonction de création des catégories 1 et 2
panel_solde_df = build_categories(panel_solde_df, grade_col="GRADE", grade2_col="GRADE 2")

# Aperçu
panel_solde_df[["GRADE","CATEGORIE_1","GRADE 2","CATEGORIE_2"]].head()


# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE", "CATEGORIE_1"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE"] != "") & (unique_grades["CATEGORIE_1"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades.xlsx", index=False)


# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE 2", "CATEGORIE_2"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE 2"] != "") & (unique_grades["CATEGORIE_2"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades_2.xlsx", index=False)

# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["CATEGORIE_1"],   # Variable en ligne
    panel_solde_df["CATEGORIE_2"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
tableau_croise_effectifs

print(((panel_solde_df["CATEGORIE_1"].isna()) | (panel_solde_df["CATEGORIE_1"].str.strip() == "")).sum())
print(((panel_solde_df["CATEGORIE_2"].isna()) | (panel_solde_df["CATEGORIE_2"].str.strip() == "")).sum())

# Filtrer les lignes
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]

# Sélectionner les colonnes à exporter
colonnes_a_garder = ["MATRICULE", "PERIODE", "CATEGORIE_1", "CATEGORIE_2", "EMPLOI_NORM"]
export_df = filtre[colonnes_a_garder]

# 💾 Exporter vers Excel
nom_fichier = "filtre_non_fonctionnaire_grade_A.xlsx"
export_df.to_excel(nom_fichier, index=False)

print(f"✅ Fichier Excel créé : {nom_fichier}")


# Appliquer la fonction de création de la variable statut
panel_solde_df, rep = build_statut_from_cats(panel_solde_df)
rep

filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["STATUT"].value_counts()
print(tab_statut)

# Tabulation simple sur STATUT
tab_statut = panel_solde_df["STATUT"].value_counts(dropna=False).sort_index()

# Total lignes
total_lignes = tab_statut.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut": tab_statut.index,
    "Effectif": tab_statut.values,
    "Proportion (%)": (tab_statut.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

# Appliquer la fonction de création du statut définitif en s'appuyant sur la période MMYYY
panel_solde_df, rep = build_statut_def_periode(panel_solde_df, return_report=True)
rep

# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut définifitif"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df


filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["Statut_def_mois"].value_counts()
print(tab_statut)


# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Vérifier si un même matricule apparaît plusieurs fois dans la même période
doublons = cas_particuliers_df[cas_particuliers_df.duplicated(subset=["MATRICULE", "PERIODE"], keep=False)]

if doublons.empty:
    print("✅ Chaque matricule est unique par période.")
else:
    print(f"⚠️ {len(doublons)} doublons trouvés (mêmes matricules dans la même période).")
    print(doublons[["MATRICULE", "PERIODE"]].sort_values(by=["MATRICULE", "PERIODE"]))
    

# Appliquer la fonction de création du statut définitif en s'appuyant sur l'année
panel_solde_df, rep = build_statut_def_annee(panel_solde_df, return_report=True)
rep

# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut_def_distribution"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df


filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A") & 
    (panel_solde_df["EMPLOI_NORM"] != "") 

]
tab_statut = filtre["Statut_def_annee"].value_counts()
print(tab_statut)

# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_annee"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_annee.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "ANNEE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")


# 1) Construire la variable propre
panel_solde_df = build_sexe_clean(
    panel_solde_df,
    id_col="MATRICULE",
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    new_col="SEXE_CLEAN"
)

# 2) Imputer à partir de SEXE_CLEAN (seul changement: sex_col="SEXE_CLEAN")
panel_solde_df, report = imputer_sexe(
    panel_solde_df,
    sex_col="SEXE_CLEAN",         # ← avant: "SEXE"
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
)

report



# Regrouper par matricule et compter le nombre de sexes distincts
sexe_par_matricule = (
    panel_solde_df.groupby("MATRICULE")["SEXE_CLEAN"]
    .nunique()
    .reset_index(name="nb_sexes_distincts")
)

# Filtrer ceux qui ont plus d’un sexe (incohérents)
problemes_sexe = sexe_par_matricule[sexe_par_matricule["nb_sexes_distincts"] > 1]

# Afficher les matricules problématiques
print("Matricules ayant plusieurs sexes enregistrés :")
print(problemes_sexe)



# 0) S'assurer qu'on a SEXE_IMPUTE
if "SEXE_IMPUTE" not in panel_solde_df.columns:
    panel_solde_df, _ = imputer_sexe(
        panel_solde_df,
        sex_col="SEXE",
        collect_col="DATE_COLLECTE",
        sex_valid_col="SEXE_IMPUTE",
        debug=False
    )

# 1) Calcul des âges (build_age_valide ne renvoie que le df)
panel_solde_df = build_age_valide(
    panel_solde_df,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    age_min=18, age_max=70,
    do_impute_age=True,
    stabilize_age=True,           # ← active AGE_IMPUTE_STABLE
    stabilize_method="last"       # 'last' recommandé; sinon 'mode' ou 'median'
)

# 2) Reconstruire les tableaux "rep" à partir du df
tab_sexe_avant      = panel_solde_df["SEXE"].value_counts(dropna=False).sort_index()
tab_sexe_avant_pct  = panel_solde_df["SEXE"].value_counts(normalize=True, dropna=False).sort_index() * 100
tab_sexe_apres      = panel_solde_df["SEXE_IMPUTE"].value_counts(dropna=False).sort_index()
tab_sexe_apres_pct  = panel_solde_df["SEXE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index() * 100
crosstab_sexe       = pd.crosstab(panel_solde_df["SEXE"], panel_solde_df["SEXE_IMPUTE"], dropna=False, margins=True, margins_name="Total")

rep = {
    "tab_sexe_avant": tab_sexe_avant,
    "tab_sexe_avant_pct": tab_sexe_avant_pct,
    "tab_sexe_apres": tab_sexe_apres,
    "tab_sexe_apres_pct": tab_sexe_apres_pct,
    "crosstab_sexe": crosstab_sexe
}

# 3) Afficher les tableaux
print(pd.DataFrame({"Effectif": rep["tab_sexe_avant"], "Pourcentage (%)": rep["tab_sexe_avant_pct"]}))
print(pd.DataFrame({"Effectif": rep["tab_sexe_apres"],  "Pourcentage (%)": rep["tab_sexe_apres_pct"]}))
print(rep["crosstab_sexe"])

# 4) Aperçu des colonnes ajoutées
cols_preview = [
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
    "ANNEE_COLLECTE","MOIS_COLLECTE",
    "SEXE","SEXE_IMPUTE",
    "AGE","AGE_VALIDE","AGE_IMPUTE"
]
print(panel_solde_df[cols_preview].head())



# Étape 1 : Supprimer les doublons sur la colonne AGE
filtered_df = panel_solde_df.drop_duplicates(subset=["AGE"])

# Étape 2 : Définir le chemin et le nom du fichier Excel de sortie
output_file = "Age.xlsx"

# Étape 3 : Exporter uniquement les colonnes pertinentes
filtered_df[["AGE", "AGE_VALIDE", "AGE_IMPUTE"]].to_excel(output_file, index=False)

# Étape 4 : Confirmation
print(f"Fichier exporté : {output_file}")



# Compter les effectifs d'âge (en incluant les NaN)
tab_age = panel_solde_df["AGE_IMPUTE"].value_counts(dropna=False).sort_index()

# Calculer les proportions (%)
tab_age_pct = (tab_age / tab_age.sum()) * 100

# Créer le tableau final
tab_age_final = pd.DataFrame({
    "Effectif": tab_age,
    "Proportion (%)": tab_age_pct.round(2)
})

# Afficher tout le tableau, y compris les NaN
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(tab_age_final)
    
    

print(panel_solde_df["AGE_IMPUTE"].isna().sum())



panel_solde_df, rep_srv = build_age_service(
    panel_solde_df,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    age_min=18, age_max=40,
    days_per_year=365,
    age_src_priority=("AGE_IMPUTE_STABLE","AGE_IMPUTE","AGE"),
    stabilize_service=True,
    stabilize_method="mode",      # 'mode' robuste pour lisser les petits écarts
    return_tables=True,
    sample_anomalies=10,
    recompute_corrected=True
)

print("→ Source d'âge utilisée pour le service :", rep_srv.get("age_source_used"))


# c) CONTRÔLE : vérifier que l’âge au service est constant par matricule
#    (AGE_AU_SERVICE_FINAL doit avoir 1 seule valeur par matricule)

nb_multi = (
    panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
                  .nunique(dropna=True).gt(1).sum()
)
tot_mats = panel_solde_df["MATRICULE"].nunique()
pct = round(nb_multi / tot_mats * 100, 2)

if nb_multi == 0:
    print("✅ L'âge au service est constant par matricule (0 matricule multi-valeurs).")
else:
    print(f"⚠️  {nb_multi:,} matricules ({pct}%) ont plusieurs âges au service FINAL.")
    # échantillon pour audit rapide :
    ex_mats = (
        panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
                      .nunique(dropna=True).reset_index(name="n")
                      .query("n>1").head(10)["MATRICULE"]
                      .tolist()
    )
    print("Exemples:", ex_mats)



panel_solde_df_mois, rep_mois = build_anciennete_mensuelle(
    panel_solde_df,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",
    min_period="2015-01",
    max_period="2020-12",
    return_by_org=True,
    return_by_person=True,
    keep_counts=True,
    include_prewindow_base=True
)

panel_solde_df = panel_solde_df_mois



# Sauvegade de la base finale : Fusion des feuille 1

In [ ]:
# ================== PARAMÈTRES S3 POUR SAUVEGARDE PANEL 1 ==================

ENDPOINT_URL = "http://minio.mon-namespace.svc.cluster.local:80"
AWS_KEY      = "wer1Or8j7hXa3QGk2M3M"
AWS_SECRET   = "YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt"
VERIFY_SSL   = False
BUCKET       = "admindataanstat"

def get_s3_client():
    """Client S3 / MinIO standardisé pour lecture/écriture."""
    return boto3.client(
        "s3",
        endpoint_url=ENDPOINT_URL,
        aws_access_key_id=AWS_KEY,
        aws_secret_access_key=AWS_SECRET,
        verify=VERIFY_SSL,
    )

def save_parquet_to_s3(df: pd.DataFrame, object_key: str, bucket: str = None, s3_client=None, engine: str = "pyarrow"):
    """
    Sauvegarde un DataFrame en Parquet sur S3/MinIO.
    """
    if bucket is None:
        bucket = BUCKET
    if s3_client is None:
        s3_client = get_s3_client()
    
    buf = io.BytesIO()
    df.to_parquet(buf, engine=engine, index=False)
    buf.seek(0)
    s3_client.put_object(Bucket=bucket, Key=object_key, Body=buf)
    print(f"✅ Sauvegardé : s3://{bucket}/{object_key}")
    return f"s3://{bucket}/{object_key}"


s3_client = get_s3_client()
save_parquet_to_s3(
    panel_solde_df,
    object_key="Solde/fusion_1_2021_2025_Nouveau.parquet",
    bucket=BUCKET,
    s3_client=s3_client
)

